# GDUT YOLO11 · pair · 3407

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

os.environ["PYTHONHASHSEED"] = "3407"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONFAULTHANDLER"] = "1"
os.environ["YOLO_CONFIG_DIR"] = "/kaggle/working/yolo_settings"
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ultralytics==8.4.115",
        "albumentations==2.0.8",
    ]
)
CODE_DIR = Path("/kaggle/working/stage1_baseline_code")
CODE_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE_DIR))
os.environ["PYTHONPATH"] = str(CODE_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/baseline_model.py
"""Importable model construction and DDP trainers for the GDUT baselines."""

import json
import os

from torch import nn
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import LOGGER, LOCAL_RANK, RANK
from ultralytics.utils.loss import BboxLoss, v8DetectionLoss
from ultralytics.utils.torch_utils import torch_distributed_zero_first

from baseline_modules import ASPP, C2fWithDWR, DWR, ModulatedDCNv2, SBA, nwd_loss_pixels
from plain_logging import configure_plain_logging

configure_plain_logging()
REVISION = "gdut_paper_protocol_v2"
BASELINES = {"yolo11_dcrcf": "yolo11n", "yolov8_dwr_aspp_nwd": "yolov8s"}


class CIoUNWDBboxLoss(BboxLoss):
    def __init__(self, reg_max, constant=12.8, nwd_weight=0.5):
        super().__init__(reg_max)
        if not 0 <= nwd_weight <= 1 or constant <= 0:
            raise ValueError("Invalid NWD mixture or normalization constant.")
        self.constant = constant
        self.nwd_weight = nwd_weight

    def forward(
        self,
        pred_dist,
        pred_bboxes,
        anchor_points,
        target_bboxes,
        target_scores,
        target_scores_sum,
        fg_mask,
        imgsz,
        stride,
    ):
        ciou, dfl = super().forward(
            pred_dist,
            pred_bboxes,
            anchor_points,
            target_bboxes,
            target_scores,
            target_scores_sum,
            fg_mask,
            imgsz,
            stride,
        )
        # Convert feature-grid box coordinates to input-image pixels.
        pixel_scale = stride.unsqueeze(0).expand(pred_bboxes.shape[0], -1, 4)[fg_mask]
        pred_pixels = pred_bboxes[fg_mask].float() * pixel_scale
        target_pixels = target_bboxes[fg_mask].float() * pixel_scale
        quality = target_scores[fg_mask].sum(-1, keepdim=True)
        nwd = (
            nwd_loss_pixels(pred_pixels, target_pixels, self.constant) * quality
        ).sum()
        nwd = nwd / target_scores_sum
        return (1 - self.nwd_weight) * ciou + self.nwd_weight * nwd, dfl


class PaperBaselineModel(DetectionModel):
    def info(self, detailed=False, verbose=True, imgsz=640):
        """Return layer, parameter and gradient counts."""
        if not verbose:
            return
        layers = sum(not module._modules for module in self.modules())
        parameters = sum(parameter.numel() for parameter in self.parameters())
        gradients = sum(p.numel() for p in self.parameters() if p.requires_grad)
        LOGGER.info(
            f"{self.baseline_id}: {layers} layers, {parameters:,} parameters, "
            f"{gradients:,} gradients; FLOPs not profiled"
        )
        return layers, parameters, gradients, 0.0

    def init_criterion(self):
        criterion = v8DetectionLoss(self)
        if self.baseline_id == "yolov8_dwr_aspp_nwd":
            criterion.bbox_loss = CIoUNWDBboxLoss(
                self.model[-1].reg_max, self.nwd_constant, self.nwd_weight
            ).to(next(self.parameters()).device)
        return criterion


def graph_layer(module, index, inputs=-1):
    module.i, module.f = index, inputs
    module.type = type(module).__name__
    module.np = sum(parameter.numel() for parameter in module.parameters())
    return module


def module_inventory(model):
    classes = (DWR, ASPP, ModulatedDCNv2, SBA)
    return {
        cls.__name__: [
            name for name, module in model.named_modules() if isinstance(module, cls)
        ]
        for cls in classes
    }


def resize_neck_input(block, channels):
    """Adapt the input projection after replacing concat with an SBA output."""
    previous = block.cv1
    block.cv1 = type(previous)(channels, previous.conv.out_channels, 1, 1)
    return block


def apply_baseline(model, baseline_id, nwd_constant=12.8, nwd_weight=0.5):
    if hasattr(model, "baseline_id") or hasattr(model, "research_variant"):
        raise ValueError("The baseline graph has already been constructed.")
    if baseline_id not in BASELINES:
        raise ValueError(baseline_id)
    old = list(model.model)
    if baseline_id == "yolov8_dwr_aspp_nwd":
        if len(old) != 23 or old[9].__class__.__name__ != "SPPF":
            raise ValueError("Expected the standard YOLOv8s graph.")
        for index in (2, 4, 6):
            model.model[index] = graph_layer(
                C2fWithDWR(old[index]), index, old[index].f
            )
        model.model[9] = graph_layer(
            ASPP(old[9].cv1.conv.in_channels, old[9].cv2.conv.out_channels), 9
        )
    else:
        if len(old) != 24 or old[10].__class__.__name__ != "C2PSA":
            raise ValueError("Expected the standard YOLO11 P3/P4/P5 graph.")
        channels = {i: old[i].cv2.conv.out_channels for i in (4, 6, 8)}
        for index in (6, 8):
            old[index] = graph_layer(
                ModulatedDCNv2(old[index].cv1.conv.in_channels, channels[index]), index
            )
        p3, p4, p5 = channels[4], channels[6], channels[8]
        n3 = old[16].cv2.conv.out_channels
        n4 = old[13].cv2.conv.out_channels
        n5 = old[22].cv2.conv.out_channels
        layers = old[:11] + [
            graph_layer(SBA(p5, p4, n4), 11, [10, 6]),
            graph_layer(resize_neck_input(old[13], n4), 12),
            graph_layer(SBA(n4, p3, n3), 13, [12, 4]),
            graph_layer(resize_neck_input(old[16], n3), 14),
            graph_layer(SBA(n3, n4, n4), 15, [14, 12]),
            graph_layer(resize_neck_input(old[19], n4), 16),
            graph_layer(SBA(n4, p5, n5), 17, [16, 10]),
            graph_layer(resize_neck_input(old[22], n5), 18),
            graph_layer(old[23], 19, [14, 16, 18]),
        ]
        model.model = nn.Sequential(*layers)
        model.save = sorted(
            {
                j
                for layer in layers
                for j in ([layer.f] if isinstance(layer.f, int) else layer.f)
                if j != -1
            }
        )
    model.__class__ = PaperBaselineModel
    model.baseline_id = baseline_id
    model.baseline_revision = REVISION
    model.nwd_constant, model.nwd_weight = nwd_constant, nwd_weight
    model.reconstruction_status = "paper_based_reimplementation_not_author_code"
    assert_inventory(model)
    return model


def assert_inventory(model):
    expected = {
        "yolo11_dcrcf": {"DWR": 0, "ASPP": 0, "ModulatedDCNv2": 2, "SBA": 4},
        "yolov8_dwr_aspp_nwd": {"DWR": 3, "ASPP": 1, "ModulatedDCNv2": 0, "SBA": 0},
    }[model.baseline_id]
    actual = {key: len(value) for key, value in module_inventory(model).items()}
    if actual != expected:
        raise RuntimeError(f"Architecture mismatch: {actual} != {expected}")
    if list(model.stride.cpu().tolist()) != [8, 16, 32]:
        raise RuntimeError("Detection strides must remain 8/16/32.")


class PaperBaselineTrainer(DetectionTrainer):
    baseline_id = "yolo11_dcrcf"

    def final_eval(self):
        """Finalize checkpoint files; the launcher validates them after DDP exits."""
        from ultralytics.engine.trainer import strip_optimizer

        with torch_distributed_zero_first(LOCAL_RANK):
            if RANK in {-1, 0}:
                latest = strip_optimizer(self.last) if self.last.exists() else {}
                if not self.best.exists():
                    raise FileNotFoundError(
                        f"No best checkpoint was saved: {self.best}"
                    )
                strip_optimizer(
                    self.best, updates={"train_results": latest.get("train_results")}
                )
                marker = dict(
                    baseline_id=self.baseline_id,
                    revision=REVISION,
                    epochs_completed=self.epoch + 1,
                    best_epoch=getattr(self.stopper, "best_epoch", None),
                    final_validation="deferred_to_single_process_fp32",
                )
                (self.save_dir / "training_complete.json").write_text(
                    json.dumps(marker, indent=2), encoding="utf-8"
                )
                LOGGER.info("Checkpoints saved. Final validation will run outside DDP.")

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = self.set_model_names_for_load(
            DetectionModel(
                cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=verbose
            )
        )
        if weights is not None:
            raise ValueError(
                "This protocol trains a new baseline from YAML without Stage1 weights."
            )
        options = json.loads(os.environ.get("GDUT_BASELINE_OPTIONS", "{}"))
        apply_baseline(model, self.baseline_id, **options)
        report = {
            "baseline_id": self.baseline_id,
            "revision": REVISION,
            "rank": os.environ.get("RANK", "-1"),
            "nc": self.data["nc"],
            "modules": module_inventory(model),
            "strides": model.stride.tolist(),
            "nwd_constant": model.nwd_constant,
            "nwd_weight": model.nwd_weight,
            "box_loss": "CIoU+NWD" if self.baseline_id.startswith("yolov8") else "CIoU",
            "parameters": sum(p.numel() for p in model.parameters()),
        }
        self.save_dir.mkdir(parents=True, exist_ok=True)
        (self.save_dir / f"architecture_rank_{report['rank']}.json").write_text(
            json.dumps(report, indent=2), encoding="utf-8"
        )
        print(json.dumps(report, indent=2), flush=True)
        return model


class DCRCFTrainer(PaperBaselineTrainer):
    baseline_id = "yolo11_dcrcf"


class DWRASPPTrainer(PaperBaselineTrainer):
    baseline_id = "yolov8_dwr_aspp_nwd"


TRAINERS = {"yolo11_dcrcf": DCRCFTrainer, "yolov8_dwr_aspp_nwd": DWRASPPTrainer}


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/baseline_modules.py
"""DCNv2, selective boundary aggregation, DWR and ASPP modules."""

import torch
from torch import nn
from torch.nn import functional as F
from torchvision.ops import deform_conv2d


class ConvBNReLU(nn.Sequential):
    def __init__(self, c1, c2, kernel=1, dilation=1, groups=1):
        padding = dilation * (kernel - 1) // 2
        super().__init__(
            nn.Conv2d(
                c1,
                c2,
                kernel,
                padding=padding,
                dilation=dilation,
                groups=groups,
                bias=False,
            ),
            nn.BatchNorm2d(c2),
            nn.ReLU(inplace=False),
        )


class DWR(nn.Module):
    """Song et al., Figure 2: 2c -> 3c -> (2c,c,c) -> 2c, residual."""

    def __init__(self, channels):
        super().__init__()
        if channels % 2:
            raise ValueError("DWR requires an even channel count.")
        half = channels // 2
        self.region = ConvBNReLU(channels, 3 * half, kernel=3)
        self.branches = nn.ModuleList(
            [
                nn.Conv2d(half, channels, 3, padding=1, groups=half, bias=False),
                nn.Conv2d(
                    half, half, 3, padding=3, dilation=3, groups=half, bias=False
                ),
                nn.Conv2d(
                    half, half, 3, padding=5, dilation=5, groups=half, bias=False
                ),
            ]
        )
        self.norm = nn.BatchNorm2d(2 * channels)
        self.project = nn.Conv2d(2 * channels, channels, 1, bias=False)

    def forward(self, x):
        groups = self.region(x).chunk(3, dim=1)
        features = torch.cat([branch(t) for branch, t in zip(self.branches, groups)], 1)
        return x + self.project(self.norm(features))


class C2fWithDWR(nn.Module):
    def __init__(self, c2f):
        super().__init__()
        self.c2f = c2f
        self.dwr = DWR(c2f.cv2.conv.out_channels)

    def forward(self, x):
        return self.dwr(self.c2f(x))


class ASPP(nn.Module):
    """Song et al., Figure 3: rates 6/12/18 and image-level pooling."""

    def __init__(self, c1, c2, hidden=256):
        super().__init__()
        self.branches = nn.ModuleList(
            [
                ConvBNReLU(c1, hidden),
                ConvBNReLU(c1, hidden, kernel=3, dilation=6),
                ConvBNReLU(c1, hidden, kernel=3, dilation=12),
                ConvBNReLU(c1, hidden, kernel=3, dilation=18),
            ]
        )
        # Global pooling branch: 1x1 convolution followed by ReLU.
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c1, hidden, 1, bias=True),
            nn.ReLU(inplace=False),
        )
        self.project = ConvBNReLU(5 * hidden, c2)

    def forward(self, x):
        pooled = F.interpolate(
            self.pool(x), size=x.shape[-2:], mode="bilinear", align_corners=False
        )
        return self.project(
            torch.cat([branch(x) for branch in self.branches] + [pooled], 1)
        )


class ModulatedDCNv2(nn.Module):
    """A modulated 3x3 deformable convolution with learned offsets and masks."""

    def __init__(self, c1, c2):
        super().__init__()
        self.offset_mask = nn.Conv2d(c1, 27, 3, padding=1)
        self.weight = nn.Parameter(torch.empty(c2, c1, 3, 3))
        self.norm = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()
        nn.init.kaiming_uniform_(self.weight, a=5**0.5)
        nn.init.zeros_(self.offset_mask.weight)
        nn.init.zeros_(self.offset_mask.bias)

    def forward(self, x):
        original_dtype = x.dtype
        # Run the compiled deformable operator in float32 under AMP and FP16 validation.
        with torch.autocast(device_type=x.device.type, enabled=False):
            features = x.float()
            fields = F.conv2d(
                features,
                self.offset_mask.weight.float(),
                self.offset_mask.bias.float(),
                padding=1,
            )
            offset, mask_logits = fields[:, :18], fields[:, 18:]
            result = deform_conv2d(
                features,
                offset,
                self.weight.float(),
                padding=(1, 1),
                mask=mask_logits.sigmoid(),
            )
        return self.act(self.norm(result.to(original_dtype)))


class RecalibrationUnit(nn.Module):
    """Recalibrate two feature tensors with 3x3 sigmoid gates."""

    def __init__(self, channels):
        super().__init__()
        self.gate_a = nn.Conv2d(channels, channels, 3, padding=1)
        self.gate_b = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, a, b):
        ga, gb = self.gate_a(a).sigmoid(), self.gate_b(b).sigmoid()
        return a * ga + (1.0 - ga) * b * gb


class SBA(nn.Module):
    """Two RAUs and a 1x1 projection; output resolution follows the second input."""

    def __init__(self, c1, c2, out_channels):
        super().__init__()
        self.first = nn.Conv2d(c1, out_channels, 1)
        self.second = nn.Conv2d(c2, out_channels, 1)
        self.rau_ab = RecalibrationUnit(out_channels)
        self.rau_ba = RecalibrationUnit(out_channels)
        self.project = nn.Conv2d(2 * out_channels, out_channels, 1)

    def forward(self, inputs):
        first, second = inputs
        a, b = self.first(first), self.second(second)
        a = F.interpolate(a, size=b.shape[-2:], mode="bilinear", align_corners=False)
        return self.project(torch.cat([self.rau_ab(a, b), self.rau_ba(b, a)], 1))


def nwd_loss_pixels(predicted, target, constant):
    """Gaussian Wasserstein box loss; xyxy coordinates and C use input-image pixels."""
    if constant <= 0:
        raise ValueError("NWD normalization constant must be positive.")
    pred_center = (predicted[..., :2] + predicted[..., 2:]) / 2
    true_center = (target[..., :2] + target[..., 2:]) / 2
    pred_half_size = (predicted[..., 2:] - predicted[..., :2]) / 2
    true_half_size = (target[..., 2:] - target[..., :2]) / 2
    delta = torch.cat([pred_center - true_center, pred_half_size - true_half_size], -1)
    distance = torch.linalg.vector_norm(delta.float(), dim=-1, keepdim=True)
    return -torch.expm1(-distance / constant)


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/baseline_preflight.py
"""Synthetic checks for the baseline graph, losses, and checkpoint portability."""

import copy
import io
from types import SimpleNamespace

import torch
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import DEFAULT_CFG_DICT

from baseline_model import (
    BASELINES,
    CIoUNWDBboxLoss,
    apply_baseline,
    assert_inventory,
    module_inventory,
)
from baseline_modules import ModulatedDCNv2, nwd_loss_pixels


def check_nwd():
    identical = torch.tensor([[0.0, 0.0, 10.0, 10.0]], requires_grad=True)
    loss = nwd_loss_pixels(identical, identical.detach(), 12.8)
    torch.testing.assert_close(loss, torch.zeros_like(loss))
    loss.sum().backward()
    assert torch.isfinite(identical.grad).all()
    predicted = torch.tensor([[1.0, 0.0, 11.0, 10.0]], requires_grad=True)
    distance = nwd_loss_pixels(predicted, identical.detach(), 12.8)
    expected = 1 - torch.exp(torch.tensor(-1 / 12.8))
    torch.testing.assert_close(distance.squeeze(), expected)
    distance.sum().backward()
    assert predicted.grad[:, [0, 2]].sum() > 0

    # The same physical box pair at different pyramid strides must have equal NWD.
    pred_pixels = torch.tensor([[[1.0, 0.0, 11.0, 10.0], [1.0, 0.0, 11.0, 10.0]]])
    true_pixels = torch.tensor([[[0.0, 0.0, 10.0, 10.0], [0.0, 0.0, 10.0, 10.0]]])
    stride = torch.tensor([[8.0], [32.0]])
    criterion = CIoUNWDBboxLoss(1, constant=12.8, nwd_weight=1.0)
    computed, _ = criterion(
        torch.zeros(1, 2, 4),
        pred_pixels / stride,
        torch.zeros(2, 2),
        true_pixels / stride,
        torch.ones(1, 2, 5) / 5,
        torch.tensor(2.0),
        torch.ones(1, 2, dtype=torch.bool),
        torch.tensor([640.0, 640.0]),
        stride,
    )
    torch.testing.assert_close(computed, expected)


def check_deformable_operator(device="cpu"):
    module = ModulatedDCNv2(4, 8).to(device)
    features = torch.rand(2, 4, 8, 8, device=device, requires_grad=True)
    module(features).square().mean().backward()
    assert module.offset_mask.weight.grad is not None
    assert torch.isfinite(module.offset_mask.weight.grad).all()
    assert module.offset_mask.weight.grad.abs().sum() > 0
    if device != "cpu":
        module.eval().half()
        with torch.no_grad():
            output = module(features.detach().half())
        assert output.dtype == torch.float16 and torch.isfinite(output).all()


def check_graph(baseline_id):
    torch.manual_seed(3407)
    model = DetectionModel(BASELINES[baseline_id] + ".yaml", nc=5, verbose=False)
    model.args = SimpleNamespace(**DEFAULT_CFG_DICT)
    apply_baseline(model, baseline_id)
    assert_inventory(model)
    model.train()
    image = torch.rand(2, 3, 64, 64)
    batch = dict(
        img=image,
        batch_idx=torch.tensor([0.0, 1.0]),
        cls=torch.tensor([[0.0], [4.0]]),
        bboxes=torch.tensor([[0.5, 0.5, 0.3, 0.3], [0.4, 0.6, 0.2, 0.3]]),
    )
    loss, _ = model(batch)
    if not torch.isfinite(loss).all():
        raise RuntimeError("Synthetic training loss is nonfinite.")
    loss.sum().backward()
    missing = [
        name
        for name, p in model.named_parameters()
        if p.requires_grad and p.grad is None
    ]
    if missing:
        raise RuntimeError(f"Modules disconnected from the training loss: {missing}")
    model.eval()
    with torch.no_grad():
        expected = model(image)[0]
    assert expected.shape == (2, 9, 84)
    stream = io.BytesIO()
    torch.save(model, stream)
    stream.seek(0)
    restored = torch.load(stream, weights_only=False).eval()
    with torch.no_grad():
        torch.testing.assert_close(restored(image)[0], expected)
    fused = copy.deepcopy(restored).fuse(verbose=False)
    with torch.no_grad():
        torch.testing.assert_close(fused(image)[0], expected, atol=2e-4, rtol=1e-4)
        single = fused(torch.rand(1, 3, 640, 640))[0]
    assert single.shape == (1, 9, 8400)
    return dict(
        modules=module_inventory(model),
        parameters=sum(p.numel() for p in model.parameters()),
    )


def run_preflight(baseline_id=None, operator_device="cpu"):
    original_threads = torch.get_num_threads()
    try:
        torch.set_num_threads(2)
        check_nwd()
        selected = [baseline_id] if baseline_id else list(BASELINES)
        if "yolo11_dcrcf" in selected:
            check_deformable_operator(operator_device)
        reports = {key: check_graph(key) for key in selected}
        print("Baseline preflight passed:", reports, flush=True)
        return reports
    finally:
        torch.set_num_threads(original_threads)


if __name__ == "__main__":
    run_preflight()


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/custom_ablation_modules.py
"""CoordConv, RepConv, routed attention, and Focal-EIoU modules."""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


def autopad(k: int, p: int | None = None, d: int = 1) -> int:
    """Return padding that preserves spatial shape for odd kernels."""
    if p is None:
        p = d * (k - 1) // 2
    return p


class ConvBNAct(nn.Module):
    """Small Conv-BN-SiLU block compatible with YOLO-style modules."""

    def __init__(
        self,
        c1: int,
        c2: int,
        k: int = 1,
        s: int = 1,
        p: int | None = None,
        g: int = 1,
        d: int = 1,
        act: bool = True,
    ) -> None:
        super().__init__()
        self.conv = nn.Conv2d(
            c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False
        )
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.bn(self.conv(x)))


class CoordConv(nn.Module):
    """
    CoordConv layer: append normalized x/y coordinate channels before a conv.

    """

    def __init__(
        self, c1: int, c2: int, k: int = 3, s: int = 1, with_r: bool = False
    ) -> None:
        super().__init__()
        self.with_r = with_r
        extra = 3 if with_r else 2
        self.conv = ConvBNAct(c1 + extra, c2, k=k, s=s)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, _, h, w = x.shape
        yy = (
            torch.linspace(-1.0, 1.0, h, device=x.device, dtype=x.dtype)
            .view(1, 1, h, 1)
            .expand(b, 1, h, w)
        )
        xx = (
            torch.linspace(-1.0, 1.0, w, device=x.device, dtype=x.dtype)
            .view(1, 1, 1, w)
            .expand(b, 1, h, w)
        )
        coords = [xx, yy]
        if self.with_r:
            rr = torch.sqrt(torch.clamp(xx.square() + yy.square(), min=0.0))
            coords.append(rr)
        return self.conv(torch.cat([x, *coords], dim=1))


class RepConv(nn.Module):
    """
    RepConv block with 3x3, 1x1, and optional identity branches during training.

    Call `switch_to_deploy()` before ONNX/TensorRT export. The deploy path is a
    single 3x3 Conv2d plus activation.
    """

    def __init__(
        self,
        c1: int,
        c2: int,
        k: int = 3,
        s: int = 1,
        deploy: bool = False,
        act: bool = True,
    ) -> None:
        super().__init__()
        assert k == 3, "This RepConv implementation fuses to a 3x3 kernel."
        self.deploy = deploy
        self.in_channels = c1
        self.out_channels = c2
        self.stride = s
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

        if deploy:
            self.rbr_reparam = nn.Conv2d(c1, c2, 3, s, 1, bias=True)
        else:
            self.rbr_dense = nn.Sequential(
                nn.Conv2d(c1, c2, 3, s, 1, bias=False),
                nn.BatchNorm2d(c2),
            )
            self.rbr_1x1 = nn.Sequential(
                nn.Conv2d(c1, c2, 1, s, 0, bias=False),
                nn.BatchNorm2d(c2),
            )
            self.rbr_identity = nn.BatchNorm2d(c1) if c1 == c2 and s == 1 else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.deploy:
            return self.act(self.rbr_reparam(x))
        out = self.rbr_dense(x) + self.rbr_1x1(x)
        if self.rbr_identity is not None:
            out = out + self.rbr_identity(x)
        return self.act(out)

    @staticmethod
    def _fuse_conv_bn(
        branch: nn.Sequential | nn.BatchNorm2d, channels: int
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if isinstance(branch, nn.Sequential):
            conv = branch[0]
            bn = branch[1]
            kernel = conv.weight
        else:
            bn = branch
            input_dim = channels
            kernel = torch.zeros(
                (input_dim, input_dim, 3, 3),
                device=bn.weight.device,
                dtype=bn.weight.dtype,
            )
            for i in range(input_dim):
                kernel[i, i, 1, 1] = 1.0

        std = torch.sqrt(bn.running_var + bn.eps)
        t = (bn.weight / std).reshape(-1, 1, 1, 1)
        fused_kernel = kernel * t
        fused_bias = bn.bias - bn.running_mean * bn.weight / std
        return fused_kernel, fused_bias

    @staticmethod
    def _pad_1x1_to_3x3(kernel: torch.Tensor) -> torch.Tensor:
        if kernel.size(2) == 3:
            return kernel
        return F.pad(kernel, [1, 1, 1, 1])

    def get_equivalent_kernel_bias(self) -> tuple[torch.Tensor, torch.Tensor]:
        k3, b3 = self._fuse_conv_bn(self.rbr_dense, self.out_channels)
        k1, b1 = self._fuse_conv_bn(self.rbr_1x1, self.out_channels)
        if self.rbr_identity is None:
            kid = torch.zeros_like(k3)
            bid = torch.zeros_like(b3)
        else:
            kid, bid = self._fuse_conv_bn(self.rbr_identity, self.out_channels)
        return k3 + self._pad_1x1_to_3x3(k1) + kid.to(k3.device), b3 + b1 + bid.to(
            b3.device
        )

    def switch_to_deploy(self) -> None:
        if self.deploy:
            return
        kernel, bias = self.get_equivalent_kernel_bias()
        self.rbr_reparam = nn.Conv2d(
            self.in_channels,
            self.out_channels,
            3,
            self.stride,
            1,
            bias=True,
        )
        self.rbr_reparam.weight.data = kernel.detach().clone()
        self.rbr_reparam.bias.data = bias.detach().clone()
        del self.rbr_dense
        del self.rbr_1x1
        if hasattr(self, "rbr_identity"):
            del self.rbr_identity
        self.deploy = True


class BiFormerBlockLite(nn.Module):
    """
    Region routing and fine token attention with a residual connection.

    Region-level routing selects the key/value tokens used by fine attention.
    The residual block uses four heads and eight-pixel regions by default.
    """

    def __init__(
        self, channels: int, num_heads: int = 4, region_size: int = 8, topk: int = 4
    ) -> None:
        super().__init__()
        assert channels % num_heads == 0, "channels must be divisible by num_heads"
        self.channels = channels
        self.num_heads = num_heads
        self.region_size = region_size
        self.topk = topk
        self.qkv = nn.Conv2d(channels, channels * 3, 1, bias=False)
        self.proj = nn.Conv2d(channels, channels, 1, bias=False)
        self.norm = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        rs = self.region_size
        pad_h = (rs - h % rs) % rs
        pad_w = (rs - w % rs) % rs
        x_pad = F.pad(x, (0, pad_w, 0, pad_h))
        hp, wp = x_pad.shape[-2:]
        gh, gw = hp // rs, wp // rs

        q, k, v = self.qkv(x_pad).chunk(3, dim=1)
        q_regions = q.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()
        k_regions = k.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()
        v_regions = v.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()

        # [B, C, GH, GW, RS, RS] -> [B, R, T, C]
        q_tokens = q_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)
        k_tokens = k_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)
        v_tokens = v_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)

        q_region = q_tokens.mean(dim=2)
        k_region = k_tokens.mean(dim=2)
        route_logits = torch.matmul(q_region, k_region.transpose(-1, -2)) / (c**0.5)
        topk = min(self.topk, gh * gw)
        route_idx = route_logits.topk(topk, dim=-1).indices

        out_regions = []
        head_dim = c // self.num_heads
        for region_idx in range(gh * gw):
            selected = route_idx[:, region_idx]
            k_sel = torch.stack(
                [
                    k_tokens[bi, selected[bi]].reshape(topk * rs * rs, c)
                    for bi in range(b)
                ],
                dim=0,
            )
            v_sel = torch.stack(
                [
                    v_tokens[bi, selected[bi]].reshape(topk * rs * rs, c)
                    for bi in range(b)
                ],
                dim=0,
            )
            q_cur = q_tokens[:, region_idx]

            qh = q_cur.reshape(b, rs * rs, self.num_heads, head_dim).transpose(1, 2)
            kh = k_sel.reshape(b, topk * rs * rs, self.num_heads, head_dim).transpose(
                1, 2
            )
            vh = v_sel.reshape(b, topk * rs * rs, self.num_heads, head_dim).transpose(
                1, 2
            )
            attn = torch.softmax(
                torch.matmul(qh, kh.transpose(-1, -2)) / (head_dim**0.5), dim=-1
            )
            out = torch.matmul(attn, vh).transpose(1, 2).reshape(b, rs * rs, c)
            out_regions.append(out)

        y = torch.stack(out_regions, dim=1).reshape(b, gh, gw, rs, rs, c)
        y = y.permute(0, 5, 1, 3, 2, 4).reshape(b, c, hp, wp)
        y = y[:, :, :h, :w]
        return x + self.norm(self.proj(y))


def xywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    x, y, w, h = boxes.unbind(-1)
    half_w = w / 2
    half_h = h / 2
    return torch.stack((x - half_w, y - half_h, x + half_w, y + half_h), dim=-1)


def focal_eiou_loss(
    pred_boxes: torch.Tensor,
    target_boxes: torch.Tensor,
    xywh: bool = True,
    gamma: float = 0.5,
    eps: float = 1e-7,
    reduction: str = "mean",
) -> torch.Tensor:
    """
    Focal-EIoU box regression loss.

    Args:
        pred_boxes: [N, 4] predicted boxes.
        target_boxes: [N, 4] target boxes.
        xywh: True when boxes are center-x, center-y, width, height.
        gamma: focal weighting exponent.
    """
    if xywh:
        pred = xywh_to_xyxy(pred_boxes)
        target = xywh_to_xyxy(target_boxes)
    else:
        pred = pred_boxes
        target = target_boxes

    px1, py1, px2, py2 = pred.unbind(-1)
    tx1, ty1, tx2, ty2 = target.unbind(-1)
    pw = (px2 - px1).clamp(min=eps)
    ph = (py2 - py1).clamp(min=eps)
    tw = (tx2 - tx1).clamp(min=eps)
    th = (ty2 - ty1).clamp(min=eps)

    inter_x1 = torch.maximum(px1, tx1)
    inter_y1 = torch.maximum(py1, ty1)
    inter_x2 = torch.minimum(px2, tx2)
    inter_y2 = torch.minimum(py2, ty2)
    inter = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
    union = pw * ph + tw * th - inter + eps
    iou = (inter / union).clamp(min=eps, max=1.0)

    pcx = (px1 + px2) / 2
    pcy = (py1 + py2) / 2
    tcx = (tx1 + tx2) / 2
    tcy = (ty1 + ty2) / 2
    center_dist = (pcx - tcx).square() + (pcy - tcy).square()

    cw = (torch.maximum(px2, tx2) - torch.minimum(px1, tx1)).clamp(min=eps)
    ch = (torch.maximum(py2, ty2) - torch.minimum(py1, ty1)).clamp(min=eps)
    c2 = cw.square() + ch.square() + eps

    eiou = (
        1.0
        - iou
        + center_dist / c2
        + (pw - tw).square() / (cw.square() + eps)
        + (ph - th).square() / (ch.square() + eps)
    )
    loss = iou.pow(gamma) * eiou

    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    if reduction == "none":
        return loss
    raise ValueError(f"Unsupported reduction: {reduction}")


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/gdut_data.py
"""Load the existing GDUT split without generating or augmenting samples."""

import hashlib
import json
from pathlib import Path

import yaml

NAMES = ["blue", "white", "yellow", "red", "none"]


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def prepared_dataset(explicit=""):
    candidates = (
        [Path(explicit)]
        if explicit
        else [p.parent for p in Path("/kaggle/input").rglob("dataset_manifest.json")]
    )
    found = []
    for root in candidates:
        path = root / "dataset_manifest.json"
        if not path.is_file():
            continue
        meta = json.loads(path.read_text(encoding="utf-8"))
        if meta.get("dataset") == "gdut":
            found.append((root, meta, sha256(path)))
    if len(found) != 1:
        raise ValueError("Mount one GDUT preparation output or set PREPARED_DIR.")
    root, meta, fingerprint = found[0]
    if meta["names"] != NAMES:
        raise ValueError(
            "Expected the GDUT class order: blue, white, yellow, red, none."
        )
    if (
        meta.get("preparation_version", 1) >= 2
        and not (root / "preparation_complete.json").exists()
    ):
        raise ValueError("The preparation output is incomplete.")
    for split in ("train", "val", "test"):
        for kind in ("images", "labels"):
            if not (root / "base" / kind / split).is_dir():
                raise FileNotFoundError(root / "base" / kind / split)
    return root, meta, fingerprint


def data_yaml(root, output_directory):
    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)
    config = dict(
        path=str(root),
        names=dict(enumerate(NAMES)),
        train="base/images/train",
        val="base/images/val",
        test="base/images/test",
    )
    target = output_directory / "gdut.yaml"
    target.write_text(yaml.safe_dump(config), encoding="utf-8")
    return str(target)


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/plain_logging.py
"""Plain-text formatting for Ultralytics console messages."""

import logging
import re


ANSI_ESCAPE = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
DECORATIONS = re.compile(r"[\U0001f000-\U0001faff\u2600-\u27bf\ufe0f\u200d]")
REPLACEMENTS = str.maketrans(
    {
        "\u2192": "->",
        "\u2190": "<-",
        "\u2194": "<->",
        "\u2265": ">=",
        "\u2264": "<=",
        "\u00d7": "x",
        "\u2705": "[OK]",
        "\u2713": "[OK]",
        "\u274c": "[FAIL]",
        "\u2717": "[FAIL]",
    }
)


def plain_text(message):
    """Remove console decoration while preserving labels and numeric values."""
    message = ANSI_ESCAPE.sub("", message).translate(REPLACEMENTS)
    return DECORATIONS.sub("", message)


class PlainFormatter(logging.Formatter):
    def format(self, record):
        message = plain_text(super().format(record))
        if record.levelno >= logging.WARNING:
            message = f"{record.levelname}: {message}"
        return message


def configure_plain_logging():
    """Keep the library's log levels and handlers, using plain-text output."""
    from ultralytics.utils import LOGGER

    for handler in LOGGER.handlers:
        handler.setFormatter(PlainFormatter("%(message)s"))


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/pretrained_conversion.py
"""Function-preserving Conv conversions; module topology is unchanged."""

import torch
from custom_ablation_modules import CoordConv, RepConv, BiFormerBlockLite


def copy_bn(source, target):
    target.load_state_dict(source.state_dict())
    # eps/momentum are not in a BN state_dict; omitting eps breaks equivalence.
    target.eps = source.eps
    target.momentum = source.momentum


@torch.no_grad()
def coord_from_conv(source):
    c = source.conv
    if c.groups != 1 or c.bias is not None:
        raise ValueError("Expected unfused standard Conv + BN")
    new = CoordConv(
        c.in_channels, c.out_channels, k=c.kernel_size[0], s=c.stride[0], with_r=False
    )
    new.to(device=c.weight.device, dtype=c.weight.dtype)
    new.conv.conv.weight.zero_()
    new.conv.conv.weight[:, : c.in_channels].copy_(c.weight)
    copy_bn(source.bn, new.conv.bn)
    return new


@torch.no_grad()
def rep_from_conv(source):
    c = source.conv
    if c.groups != 1 or c.bias is not None or c.kernel_size != (3, 3):
        raise ValueError("Expected unfused standard 3x3 Conv + BN")
    new = RepConv(c.in_channels, c.out_channels, k=3, s=c.stride[0], deploy=False)
    new.to(device=c.weight.device, dtype=c.weight.dtype)
    new.rbr_dense[0].weight.copy_(c.weight)
    copy_bn(source.bn, new.rbr_dense[1])
    # Keep new branches trainable; zero BN scale makes their initial contribution zero.
    new.rbr_1x1[1].weight.zero_()
    new.rbr_1x1[1].bias.zero_()
    if new.rbr_identity is not None:
        new.rbr_identity.weight.zero_()
        new.rbr_identity.bias.zero_()
    return new


@torch.no_grad()
def identity_start_attention(channels, source):
    new = BiFormerBlockLite(channels=channels, num_heads=4)
    p = next(source.parameters())
    new.to(device=p.device, dtype=p.dtype)
    new.norm.weight.zero_()
    new.norm.bias.zero_()
    # Zero residual scale starts the attention block as an identity map.
    return new


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/research_trainer.py
"""Model construction and distributed trainers for the ablation experiments."""

import json
import os

from plain_logging import configure_plain_logging

configure_plain_logging()

from torch import nn
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.loss import BboxLoss, v8DetectionLoss

from custom_ablation_modules import (
    BiFormerBlockLite,
    CoordConv,
    RepConv,
    focal_eiou_loss,
)
from pretrained_conversion import (
    coord_from_conv,
    identity_start_attention,
    rep_from_conv,
)

ARCHITECTURE_REVISION = "biformer_both_backbones_v2"
INITIALIZATION_POLICY = "preserve_pretrained_v1"


class SPPFWithBiFormer(nn.Module):
    """Retain the YOLOv8 SPPF and apply attention to its P5 output."""

    def __init__(self, sppf, attention):
        super().__init__()
        self.sppf = sppf
        self.attention = attention

    def forward(self, x):
        return self.attention(self.sppf(x))


class FocalEIoUBboxLoss(BboxLoss):
    """Use Focal-EIoU for box regression and retain the native DFL term."""

    def forward(
        self,
        pred_dist,
        pred_bboxes,
        anchor_points,
        target_bboxes,
        target_scores,
        target_scores_sum,
        fg_mask,
        imgsz,
        stride,
    ):
        _, loss_dfl = super().forward(
            pred_dist,
            pred_bboxes,
            anchor_points,
            target_bboxes,
            target_scores,
            target_scores_sum,
            fg_mask,
            imgsz,
            stride,
        )
        weight = target_scores[fg_mask].sum(-1)
        loss_box = focal_eiou_loss(
            pred_bboxes[fg_mask],
            target_bboxes[fg_mask],
            xywh=False,
            gamma=0.5,
            reduction="none",
        )
        return (loss_box * weight).sum() / target_scores_sum, loss_dfl


class ResearchDetectionModel(DetectionModel):
    def init_criterion(self):
        criterion = v8DetectionLoss(self)
        if self.research_variant in ("A4", "A6"):
            criterion.bbox_loss = FocalEIoUBboxLoss(self.model[-1].reg_max).to(
                next(self.parameters()).device
            )
        return criterion


def inventory(model):
    """Return top-level graph indices containing each custom module."""
    module_types = {
        "CoordConv": CoordConv,
        "RepConv": RepConv,
        "BiFormerBlockLite": BiFormerBlockLite,
    }
    return {
        name: [
            index
            for index, layer in enumerate(model.model)
            if any(isinstance(child, module_type) for child in layer.modules())
        ]
        for name, module_type in module_types.items()
    }


def replace(model, index, replacement):
    previous = model.model[index]
    replacement.i = getattr(previous, "i", index)
    replacement.f = getattr(previous, "f", -1)
    replacement.type = replacement.__class__.__name__
    replacement.np = sum(parameter.numel() for parameter in replacement.parameters())
    model.model[index] = replacement


def patch(model, variant, initialization=INITIALIZATION_POLICY):
    """Construct one ablation from a stock Stage1 model."""
    if variant not in ("A1", "A2", "A3", "A4", "A5", "A6"):
        raise ValueError(f"Unknown variant: {variant}")
    if initialization not in (INITIALIZATION_POLICY, "legacy_random"):
        raise ValueError(f"Unknown initialization policy: {initialization}")
    preserve = initialization == INITIALIZATION_POLICY
    classes = {layer.__class__.__name__ for layer in model.model}
    if "C2PSA" in classes:
        family = "yolo11"
    elif "C2f" in classes:
        family = "yolov8"
    else:
        raise ValueError("The ablations support YOLO11s and YOLOv8s only.")

    conversions = []
    if variant in ("A2", "A6"):
        previous = model.model[0]
        conv = previous.conv
        replacement = (
            coord_from_conv(previous)
            if preserve
            else CoordConv(conv.in_channels, conv.out_channels, k=3, s=2)
        )
        replace(model, 0, replacement)
        conversions.append(
            {"index": 0, "module": "CoordConv", "copied_conv_bn": preserve}
        )

    if variant in ("A3", "A6"):
        for index, layer in enumerate(model.model):
            if variant == "A6" and index == 0:
                continue
            conv = getattr(layer, "conv", None)
            if getattr(conv, "kernel_size", None) == (3, 3):
                replacement = (
                    rep_from_conv(layer)
                    if preserve
                    else RepConv(
                        conv.in_channels,
                        conv.out_channels,
                        k=3,
                        s=conv.stride[0],
                    )
                )
                replace(model, index, replacement)
                conversions.append(
                    {"index": index, "module": "RepConv", "copied_conv_bn": preserve}
                )

    if variant in ("A5", "A6"):
        index = 10 if family == "yolo11" else 9
        previous = model.model[index]
        expected_type = "C2PSA" if family == "yolo11" else "SPPF"
        if previous.__class__.__name__ != expected_type:
            raise ValueError(f"Expected {expected_type} at layer {index}.")
        channels = (
            previous.cv1.conv.in_channels
            if family == "yolo11"
            else previous.cv2.conv.out_channels
        )
        attention = (
            identity_start_attention(channels, previous)
            if preserve
            else BiFormerBlockLite(channels, num_heads=4)
        )
        replacement = (
            attention if family == "yolo11" else SPPFWithBiFormer(previous, attention)
        )
        replace(model, index, replacement)
        conversions.append(
            {
                "index": index,
                "module": "BiFormerBlockLite",
                "placement": "replace_C2PSA" if family == "yolo11" else "after_SPPF",
                "identity_start": preserve,
            }
        )

    rep_indices = (
        [0, 1, 3, 5, 7, 17, 20] if family == "yolo11" else [0, 1, 3, 5, 7, 16, 19]
    )
    expected = {
        "CoordConv": [0] if variant in ("A2", "A6") else [],
        "RepConv": rep_indices if variant in ("A3", "A6") else [],
        "BiFormerBlockLite": (
            [10 if family == "yolo11" else 9] if variant in ("A5", "A6") else []
        ),
    }
    if variant == "A6":
        expected["RepConv"] = expected["RepConv"][1:]
    if inventory(model) != expected:
        raise RuntimeError(f"Unexpected module inventory: {inventory(model)}")

    model.research_variant = variant
    model.research_family = family
    model.research_status = "as_defined"
    model.architecture_revision = ARCHITECTURE_REVISION
    model.initialization_policy = initialization
    model.initialization_audit = conversions
    return model


class ResearchTrainer(DetectionTrainer):
    variant = "A1"

    def get_model(self, cfg=None, weights=None, verbose=True):
        # Each DDP worker constructs the graph before loading custom weights.
        model = self.set_model_names_for_load(
            ResearchDetectionModel(
                cfg,
                nc=self.data["nc"],
                ch=self.data["channels"],
                verbose=verbose,
            )
        )
        if weights is not None and hasattr(weights, "research_variant"):
            if weights.research_variant != self.variant:
                raise ValueError("Each ablation must start from its Stage1 checkpoint.")
            if (
                weights.research_family == "yolov8"
                and self.variant in ("A5", "A6")
                and getattr(weights, "architecture_revision", None)
                != ARCHITECTURE_REVISION
            ):
                raise ValueError(
                    "This YOLOv8 checkpoint predates the BiFormer extension."
                )
            patch(
                model,
                self.variant,
                initialization=getattr(
                    weights, "initialization_policy", "legacy_random"
                ),
            )
            model.load(weights)
        else:
            if weights is not None:
                model.load(weights)
            patch(model, self.variant)

        report = {
            "variant": self.variant,
            "family": model.research_family,
            "status": model.research_status,
            "rank": os.environ.get("RANK", "-1"),
            "modules": inventory(model),
            "nc": self.data["nc"],
            "box_loss": "Focal-EIoU" if self.variant in ("A4", "A6") else "CIoU",
            "cls_loss": "BCE",
            "dfl": "native Ultralytics DFL",
            "parameters": sum(parameter.numel() for parameter in model.parameters()),
            "architecture_revision": ARCHITECTURE_REVISION,
            "initialization_policy": model.initialization_policy,
            "initialization_audit": model.initialization_audit,
        }
        self.save_dir.mkdir(parents=True, exist_ok=True)
        path = self.save_dir / f"architecture_rank_{report['rank']}.json"
        path.write_text(json.dumps(report, indent=2), encoding="utf-8")
        print(json.dumps(report, indent=2), flush=True)
        return model


class A1Trainer(ResearchTrainer):
    variant = "A1"


class A2Trainer(ResearchTrainer):
    variant = "A2"


class A3Trainer(ResearchTrainer):
    variant = "A3"


class A4Trainer(ResearchTrainer):
    variant = "A4"


class A5Trainer(ResearchTrainer):
    variant = "A5"


class A6Trainer(ResearchTrainer):
    variant = "A6"


TRAINERS = {
    trainer.variant: trainer
    for trainer in (A1Trainer, A2Trainer, A3Trainer, A4Trainer, A5Trainer, A6Trainer)
}


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/sitecustomize.py
"""Apply the notebook console format in distributed training workers."""

import os

if "LOCAL_RANK" in os.environ:
    from plain_logging import configure_plain_logging

    configure_plain_logging()


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_comparison.py
"""Compare Stage1 baseline runs and existing CLEAN A6 checkpoints on common splits."""

import argparse
import csv
import json
from pathlib import Path

from ultralytics import YOLO
import yaml

from baseline_model import assert_inventory
from gdut_data import prepared_dataset, sha256
from research_trainer import ARCHITECTURE_REVISION, inventory
from warm_data import dataset_yaml
from warm_metrics import benchmark_forward, export_metrics
from warm_protocol import COMMON, EVALUATION, MODELS, REVISION
from warm_runner import CODE_DIR, check_version, release, write_json


def csv_file(path, rows):
    with Path(path).open("w", newline="", encoding="utf-8-sig") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def inputs_for_comparison(input_root, dataset_hash, software_check=False):
    found = {}
    for path in Path(input_root).rglob("run_manifest.json"):
        value = json.loads(path.read_text())
        if value.get("dataset") != "gdut":
            continue
        if (
            value.get("warm_revision") == REVISION
            and value.get("baseline_id") in MODELS
        ):
            identifier = value["baseline_id"]
        elif value.get("stage") == "A6" and value.get("model") in (
            "yolo11s",
            "yolov8s",
        ):
            identifier = "A6_" + value["model"]
            if value.get("architecture_revision") != ARCHITECTURE_REVISION:
                raise ValueError(
                    "Use the original CLEAN A6 revision with BiFormer on both backbones."
                )
        else:
            continue
        if identifier in found:
            raise ValueError(f"Mount one output for {identifier}.")
        if value.get("smoke") and not software_check:
            raise ValueError("Use full experiment checkpoints.")
        if (
            value.get("dataset_fingerprint") != dataset_hash
            or value.get("imgsz") != 640
        ):
            raise ValueError("Dataset version or input size differs.")
        checkpoint = (path.parent / value["checkpoint"]).resolve()
        if not checkpoint.is_relative_to(path.parent.resolve()):
            raise ValueError("Checkpoints must remain inside their run directory.")
        if not checkpoint.is_file() or sha256(checkpoint) != value["checkpoint_sha256"]:
            raise ValueError("Missing or modified comparison checkpoint.")
        found[identifier] = (path, value, checkpoint)
    required = set(MODELS) | {"A6_yolo11s", "A6_yolov8s"}
    if found.keys() != required:
        raise ValueError(f"Missing inputs: {sorted(required-found.keys())}")
    baseline = found["yolov8_dwr_aspp_nwd"][1]
    reference = found["A6_yolov8s"][1]
    if baseline["source_checkpoint_sha256"] != reference["source_checkpoint_sha256"]:
        raise ValueError(
            "YOLOv8 baseline and A6 must start from the same stock Stage1 checkpoint."
        )
    left = found["yolo11_dcrcf"][1]["runtime"]
    right = baseline["runtime"]
    if left["hardware"] != right["hardware"] or left["software"] != right["software"]:
        raise ValueError("The two baseline training environments differ.")
    return found


def audit_recipe(found, software_check=False):
    rows = []
    for key, expected in COMMON.items():
        row = dict(parameter=key, configured_common=expected)
        known = []
        for identifier, (path, manifest, _) in found.items():
            saved = path.parent / "args.yaml"
            args = (
                yaml.safe_load(saved.read_text())
                if saved.is_file()
                else manifest.get("training_args", {})
            )
            value = args.get(key)
            row[identifier] = value
            if value is not None:
                known.append(value)
                if value != expected and not software_check:
                    raise ValueError(
                        f"Stage2 configuration differs: {identifier}/{key}."
                    )
        row["status"] = (
            "matched" if len(known) == len(found) else "not_recorded_in_all_runs"
        )
        rows.append(row)
    return rows


def compare(
    prepared_dir="",
    device="0",
    input_root="/kaggle/input",
    working="/kaggle/working",
    software_check=False,
):
    check_version()
    prepared, _, dataset_hash = prepared_dataset(prepared_dir)
    found = inputs_for_comparison(input_root, dataset_hash, software_check)
    recipe = audit_recipe(found, software_check)
    output = Path(working) / "stage1_baseline_comparison"
    if output.exists():
        raise FileExistsError(f"Use a fresh comparison directory: {output}")
    output.mkdir(parents=True)
    data = dataset_yaml(prepared, output / "configuration", False)
    rows = []
    for identifier, (path, metadata, weight) in sorted(found.items()):
        model = YOLO(str(weight))
        if identifier in MODELS:
            assert_inventory(model.model)
            if (
                model.model.warm_revision != REVISION
                or model.model.baseline_id != identifier
            ):
                raise ValueError("Baseline graph differs from its manifest.")
        elif inventory(model.model) != metadata["modules"]:
            raise ValueError("A6 graph differs from its manifest.")
        parameters = sum(p.numel() for p in model.model.parameters())
        results_file = path.parent / "results.csv"
        actual_epochs = metadata.get("epochs_completed")
        if actual_epochs is None and results_file.is_file():
            with results_file.open() as stream:
                actual_epochs = sum(1 for _ in csv.DictReader(stream))
        details = dict(
            manifest=metadata,
            source_manifest=str(path),
            checkpoint_sha256=sha256(weight),
            splits={},
        )
        current_rows = []
        for split in ("val", "test"):
            settings = dict(EVALUATION)
            if software_check:
                settings.update(imgsz=64, batch=2, plots=False)
            result = model.val(
                data=data,
                split=split,
                device=device,
                project=str(output),
                name=identifier + "_" + split,
                **settings,
            )
            metrics, classes, definitions = export_metrics(result)
            details["splits"][split] = dict(
                metrics=metrics, per_class=classes, definitions=definitions
            )
            current_rows.append(
                dict(
                    model=identifier,
                    backbone=metadata["model"],
                    Stage2_epochs_completed=actual_epochs,
                    Stage1_epochs_completed=metadata.get("source_manifest", {}).get(
                        "observed_epochs_completed"
                    ),
                    split=split,
                    parameters=parameters,
                    **metrics,
                )
            )
        timing = benchmark_forward(model.model, device, smoke=software_check)
        for row in current_rows:
            row.update(
                forward_ms_median=timing["forward_ms_median"],
                deployment_parameters=timing["deployment_parameters"],
            )
        rows.extend(current_rows)
        details["timing"] = timing
        write_json(output / (identifier + "_details.json"), details)
        del model
        release()
    references = json.loads((CODE_DIR / "paper_references.json").read_text())
    paper_rows = []
    for row in rows:
        if row["model"] not in references:
            continue
        paper = references[row["model"]]
        record = dict(
            model=row["model"],
            split=row["split"],
            paper_dataset=paper["dataset"],
            experiment_dataset="GDUT",
            doi=paper["doi"],
            comparison_type="cross_dataset",
        )
        for metric in ("precision", "recall", "mAP50", "mAP50_95"):
            record["paper_" + metric] = paper["metrics"][metric]
            record["experiment_" + metric] = row[metric]
            record[metric + "_delta_pp"] = 100 * (
                row[metric] - paper["metrics"][metric]
            )
        paper_rows.append(record)
    for split in ("val", "test"):
        csv_file(
            output / ("comparison_" + split + ".csv"),
            [row for row in rows if row["split"] == split],
        )
    csv_file(output / "paper_vs_experiment.csv", paper_rows)
    csv_file(output / "stage2_recipe_audit.csv", recipe)
    report = dict(
        complete=True,
        software_check=software_check,
        results=rows,
        paper_comparison=paper_rows,
        stage2_recipe=COMMON,
        dataset_fingerprint=dataset_hash,
        yolo8_same_stage1_checkpoint=True,
        yolo11_scale_comparison="YOLO11n baseline versus YOLO11s A6; separate scale-matched stock Stage1 sources",
        hardware={
            key: value[1].get("runtime", "not recorded in original manifest")
            for key, value in found.items()
        },
        selection="best validation mAP50-95; original early stopping policy",
        training_budget="Configured epochs and early stopping match; realized Stage1/Stage2 epochs can differ",
    )
    write_json(output / "comparison_report.json", report)
    lines = [
        "# Baseline theo protocol Stage1 của A6 CLEAN",
        "",
        "## 2.1. Thực nghiệm",
        "",
        "Hai baseline dùng Stage1 GDUT theo scale tương ứng, sau đó AdamW, lr 0.0001, ảnh 640, batch 16, seed 3407, tối đa 100 epoch, patience 50 và base + hardaug. Kiến trúc và loss baseline giữ nguyên. YOLOv8 dùng cùng hash Stage1 với A6; YOLO11n dùng Stage1 riêng vì A6 là YOLO11s. Số epoch thực tế và ngân sách Stage1 được ghi riêng, không coi là bằng nhau khi early stopping khác nhau.",
        "",
        "## 2.2. Paper và thực nghiệm",
        "",
        "paper_vs_experiment.csv chứa số công bố SHWD và số GDUT theo từng split, đơn vị 0–1 và delta theo điểm phần trăm. comparison_val.csv và comparison_test.csv tách kết quả của bốn mô hình. Đây là thí nghiệm chuyển trọng số theo protocol GDUT, không phải lặp nguyên recipe gốc của paper; paper YOLO-DCRCF mô tả train không dùng pretrained.",
        "",
        "## 2.3. Gap Analysis",
        "",
        "Khác biệt được xác nhận: SHWD hai lớp so với GDUT năm lớp; khởi tạo pretrained/Stage1, optimizer và augmentation đã điều chỉnh theo A6. Domain, kích thước đối tượng và phân bố lớp có thể ảnh hưởng, nhưng chưa được cô lập bằng thí nghiệm riêng. Chênh lệch được báo cáo nguyên trạng, không gán toàn bộ cho kiến trúc. Bảng audit phân biệt giá trị đã ghi với giá trị thiếu trong run cũ.",
        "",
    ]
    for row in paper_rows:
        lines.append(
            f"- {row['model']} [{row['split']}]: mAP50 {100*row['experiment_mAP50']:.2f}%, paper {100*row['paper_mAP50']:.2f}%, delta {row['mAP50_delta_pp']:+.2f} điểm phần trăm."
        )
    lines.extend(
        [
            "",
            "Nguồn: https://doi.org/10.3390/jimaging11090320 ; https://doi.org/10.1038/s41598-024-68446-z",
        ]
    )
    (output / "gap_analysis.vi.md").write_text(
        "\n".join(lines) + "\n", encoding="utf-8"
    )
    print(f"Comparison complete: {output}", flush=True)
    return report


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--prepared-dir", default="")
    parser.add_argument("--device", default="0")
    args = parser.parse_args()
    compare(args.prepared_dir, args.device)


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_data.py
"""Locate stock GDUT checkpoints and the existing train-only hardaug images."""

import csv
import json
from pathlib import Path

import yaml

from gdut_data import NAMES, sha256


def dataset_yaml(prepared, output, augmented):
    prepared, output = Path(prepared), Path(output)
    train = [str(prepared / "base/images/train")]
    if augmented:
        manifest = json.loads((prepared / "augmentation_manifest.json").read_text())
        if (
            manifest.get("image_size") != 640
            or manifest.get("copies_per_train_image") != 1
            or not manifest.get("hue_saturation_transform_removed")
            or manifest.get("validation_test_augmented")
        ):
            raise ValueError("Use the GDUT color-safe 640 train-only hardaug output.")
        base = sorted((prepared / "base/images/train").glob("*.jpg"))
        hardaug = prepared / "hardaug/images/train"
        expected = {image.stem + "_hardaug.jpg" for image in base}
        if {p.name for p in hardaug.glob("*.jpg")} != expected:
            raise ValueError("Hardaug images do not match the base training images.")
        for image in hardaug.glob("*.jpg"):
            if not (
                prepared / "hardaug/labels/train" / (image.stem + ".txt")
            ).is_file():
                raise FileNotFoundError(f"Missing hardaug label: {image.stem}")
        train.append(str(hardaug))
    output.mkdir(parents=True, exist_ok=True)
    path = output / ("gdut_aug.yaml" if augmented else "gdut_base.yaml")
    path.write_text(
        yaml.safe_dump(
            dict(
                path=str(prepared),
                names=dict(enumerate(NAMES)),
                train=train,
                val=str(prepared / "base/images/val"),
                test=str(prepared / "base/images/test"),
            )
        ),
        encoding="utf-8",
    )
    return str(path)


def find_stage1(input_root, family, dataset_hash, explicit="", allow_smoke=False):
    found = []
    for path in Path(explicit or input_root).rglob("run_manifest.json"):
        value = json.loads(path.read_text())
        if (value.get("dataset"), value.get("stage"), value.get("model")) == (
            "gdut",
            "S1",
            family,
        ):
            found.append((path, value))
    if len(found) != 1:
        raise ValueError(
            f"Mount one stock GDUT Stage1 {family} output; found {len(found)}."
        )
    path, value = found[0]
    if (
        value.get("dataset_fingerprint") != dataset_hash
        or value.get("names") != NAMES
        or value.get("imgsz") != 640
    ):
        raise ValueError("Stage1 data version, classes or image size differ.")
    if value.get("smoke") and not allow_smoke:
        raise ValueError("Use a full Stage1 checkpoint.")
    expected = dict(
        optimizer="AdamW",
        lr0=0.0003,
        momentum=0.9,
        warmup_bias_lr=0.0,
        epochs=100,
        batch=16,
        imgsz=640,
        seed=3407,
    )
    if not allow_smoke:
        for key, configured in expected.items():
            if value.get("training_args", {}).get(key) != configured:
                raise ValueError(f"Stage1 does not use the CLEAN source recipe: {key}.")
    checkpoint = (path.parent / value["checkpoint"]).resolve()
    if not checkpoint.is_relative_to(path.parent.resolve()):
        raise ValueError("Checkpoint must remain inside its run directory.")
    if not checkpoint.is_file() or sha256(checkpoint) != value["checkpoint_sha256"]:
        raise ValueError("Missing or modified Stage1 checkpoint.")
    results = path.parent / "results.csv"
    if results.is_file():
        with results.open() as stream:
            value["observed_epochs_completed"] = sum(1 for _ in csv.DictReader(stream))
    return checkpoint, value


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_environment.py
"""Runtime metadata and reproducible configuration fingerprints."""

import hashlib
import importlib
import json
import os
import platform
import torch

PIN = "8.4.115"


def fingerprint(value):
    encoded = json.dumps(value, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()


def runtime_snapshot(device, expected_gpu="Tesla T4", smoke=False):
    if device == "cpu":
        if not smoke:
            raise ValueError("Use a CUDA GPU for the full study.")
        gpus = []
    else:
        if torch.version.cuda is None:
            raise RuntimeError(
                "This session has CPU-only PyTorch. In Kaggle select Settings > "
                "Accelerator > GPU T4 x2, restart the session, then Run All."
            )
        if not torch.cuda.is_available():
            raise RuntimeError(
                "CUDA is not available in this session. Enable the Kaggle GPU "
                "accelerator and restart the session before training."
            )
        indices = [int(value) for value in str(device).split(",")]
        if len(indices) != len(set(indices)):
            raise ValueError("GPU indices must be unique.")
        count = torch.cuda.device_count()
        if any(index < 0 or index >= count for index in indices):
            raise RuntimeError(
                f"DEVICE={device!r} requests unavailable indices; this session has only "
                f"{count} GPU(s). Select GPU T4 x2 or use DEVICE='0' consistently."
            )
        gpus = []
        for index in indices:
            prop = torch.cuda.get_device_properties(index)
            if expected_gpu and expected_gpu not in prop.name:
                raise ValueError(f"Expected {expected_gpu}; found {prop.name}.")
            gpus.append(
                dict(
                    name=prop.name,
                    memory_bytes=prop.total_memory,
                    capability=[prop.major, prop.minor],
                )
            )
    versions = {}
    for name in (
        "torch",
        "torchvision",
        "ultralytics",
        "albumentations",
        "numpy",
        "scipy",
        "cv2",
        "PIL",
    ):
        try:
            versions[name] = importlib.import_module(name).__version__
        except (ImportError, AttributeError):
            if not smoke:
                raise RuntimeError(f"Install the pinned requirements; missing {name}.")
            versions[name] = "not_installed_software_check"
    if not smoke and (
        versions["ultralytics"] != PIN or versions["albumentations"] != "2.0.8"
    ):
        raise RuntimeError("Install the pinned study requirements.")
    software = dict(
        packages=versions,
        python=platform.python_version(),
        cuda=torch.version.cuda,
        cudnn=torch.backends.cudnn.version(),
        system=platform.system(),
        architecture=platform.machine(),
    )
    return dict(
        hardware=dict(gpus=gpus, cpu_architecture=platform.machine()),
        software=software,
        host=dict(cpu=platform.processor(), cpu_count=os.cpu_count()),
    )


def contract_differences(left, right, prefix=""):
    differences = []
    for key in sorted(left.keys() | right.keys()):
        path = f"{prefix}.{key}" if prefix else key
        a, b = left.get(key), right.get(key)
        if isinstance(a, dict) and isinstance(b, dict):
            differences.extend(contract_differences(a, b, path))
        elif a != b:
            differences.append(dict(field=path, left=a, right=b))
    return differences


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_launch.py
"""Stream subprocess output and preserve the complete training log."""

from collections import deque
from pathlib import Path
import subprocess


def run_logged(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    arguments = list(command)
    if "-u" not in arguments[1:]:
        arguments.insert(1, "-u")
    tail = deque(maxlen=35)
    with log_path.open("w", encoding="utf-8") as log:
        with subprocess.Popen(
            arguments,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        ) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                    tail.append(line)
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if returncode:
        raise RuntimeError(
            f"Process stopped with exit code {returncode}. Full log: {log_path}\n"
            + "".join(tail)
        )
    return log_path


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_metrics.py
"""Export AP, precision, recall and class-wise F1 with explicit confidence rules."""

import copy
import time

import numpy as np
import torch

from custom_ablation_modules import RepConv
from gdut_data import NAMES


def f1_macro(precision, recall):
    precision, recall = np.asarray(precision), np.asarray(recall)
    values = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(precision, dtype=float),
        where=precision + recall > 0,
    )
    return float(values.mean()) if values.size else 0.0


def export_metrics(result, confidence=0.25):
    box = result.box
    summary = dict(
        precision=float(box.mp),
        recall=float(box.mr),
        mAP50=float(box.map50),
        mAP50_95=float(box.map),
        F1_native_macro=float(np.asarray(box.f1).mean()) if len(box.f1) else 0.0,
    )
    per_class, p_fixed, r_fixed = [], [], []
    for index, class_id in enumerate(box.ap_class_index):
        p = float(np.interp(confidence, box.px, box.p_curve[index]))
        r = float(np.interp(confidence, box.px, box.r_curve[index]))
        p_fixed.append(p)
        r_fixed.append(r)
        per_class.append(
            dict(
                class_id=int(class_id),
                name=NAMES[class_id],
                precision=float(box.p[index]),
                recall=float(box.r[index]),
                F1_native=float(box.f1[index]),
                AP50=float(box.all_ap[index, 0]),
                AP50_95=float(box.all_ap[index].mean()),
                precision_conf025=p,
                recall_conf025=r,
                F1_conf025=f1_macro([p], [r]),
            )
        )
    present = {row["class_id"] for row in per_class}
    for class_id, name in enumerate(NAMES):
        if class_id not in present:
            per_class.append(
                dict(
                    class_id=class_id,
                    name=name,
                    precision=0.0,
                    recall=0.0,
                    F1_native=0.0,
                    AP50=0.0,
                    AP50_95=0.0,
                    precision_conf025=0.0,
                    recall_conf025=0.0,
                    F1_conf025=0.0,
                )
            )
            p_fixed.append(0.0)
            r_fixed.append(0.0)
    per_class.sort(key=lambda row: row["class_id"])
    summary.update(
        precision_conf025_macro=float(np.mean(p_fixed)),
        recall_conf025_macro=float(np.mean(r_fixed)),
        F1_conf025_macro=f1_macro(p_fixed, r_fixed),
    )
    definitions = dict(
        precision_recall="Ultralytics class means at the native operating point",
        F1_native_macro="Mean class F1 at the native mean-F1 operating point; IoU 0.5",
        F1_conf025_macro="Mean class F1 interpolated from PR/confidence curves at confidence 0.25; IoU 0.5",
        AP="Ultralytics 101-point AP integration; AP50 and mean AP at IoU 0.50:0.05:0.95",
    )
    return summary, per_class, definitions


def benchmark_forward(model, device, smoke=False):
    target = torch.device("cpu" if device == "cpu" else "cuda:" + str(device))
    measured = copy.deepcopy(model).to(target).float().eval()
    converted = 0
    for layer in list(measured.modules()):
        if isinstance(layer, RepConv):
            layer.switch_to_deploy()
            converted += 1
    measured.fuse(verbose=False)
    tensor = torch.zeros(1, 3, 640, 640, device=target)
    iterations, warmup = (2, 1) if smoke else (30, 10)

    def sync():
        if target.type == "cuda":
            torch.cuda.synchronize(target)

    times = []
    with torch.inference_mode():
        for _ in range(warmup):
            measured(tensor)
        sync()
        for _ in range(iterations):
            sync()
            start = time.perf_counter()
            measured(tensor)
            sync()
            times.append(1000 * (time.perf_counter() - start))
    return dict(
        forward_ms_median=float(np.median(times)),
        forward_ms_mean=float(np.mean(times)),
        batch=1,
        imgsz=640,
        precision="float32",
        includes_preprocessing=False,
        includes_nms=False,
        warmup=warmup,
        iterations=iterations,
        device=str(target),
        hardware=torch.cuda.get_device_name(target) if target.type == "cuda" else "CPU",
        repconv_deploy_blocks=converted,
        deployment_parameters=sum(p.numel() for p in measured.parameters()),
    )


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_model.py
"""Load stock Stage1 weights before inserting the baseline modules."""

import json
import os

import torch
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

from baseline_model import PaperBaselineTrainer, apply_baseline, module_inventory
from warm_protocol import MODELS, REVISION


def validate_stock_model(model, identifier):
    if hasattr(model, "research_variant") or hasattr(model, "baseline_id"):
        raise ValueError(
            "Use a stock Stage1 checkpoint, not an A6 or baseline checkpoint."
        )
    expected = MODELS[identifier]
    classes = {type(layer).__name__ for layer in model.model}
    family = (
        "yolo11" if "C2PSA" in classes else "yolov8" if "C2f" in classes else "unknown"
    )
    expected_family = "yolo11" if expected.startswith("yolo11") else "yolov8"
    stem = model.model[0].conv.out_channels
    if family != expected_family or stem != (16 if expected == "yolo11n" else 32):
        raise ValueError(f"Expected the {expected} stock Stage1 scale.")
    if model.model[-1].nc != 5:
        raise ValueError("Stage1 must use the five GDUT classes.")


def transfer_baseline(model, identifier):
    validate_stock_model(model, identifier)
    previous_parameters = {
        id(value): (name, value) for name, value in model.named_parameters()
    }
    previous_buffers = {
        id(value): (name, value) for name, value in model.named_buffers()
    }
    pooling = model.model[9] if identifier.startswith("yolov8") else None
    apply_baseline(model, identifier)
    explicit = {}
    if pooling is not None:
        target = model.model[9].branches[0]
        with torch.no_grad():
            target[0].weight.copy_(pooling.cv1.conv.weight)
            target[1].load_state_dict(pooling.cv1.bn.state_dict())
        target[1].eps, target[1].momentum = pooling.cv1.bn.eps, pooling.cv1.bn.momentum
        explicit["model.9.branches.0.0.weight"] = "model.9.cv1.conv.weight"
        for key in pooling.cv1.bn.state_dict():
            explicit["model.9.branches.0.1." + key] = "model.9.cv1.bn." + key
    mapping, initialized, transferred_elements = [], [], 0
    for name, parameter in model.named_parameters():
        retained = previous_parameters.get(id(parameter))
        source_name = retained[0] if retained is not None else explicit.get(name)
        if source_name is not None:
            mapping.append(
                dict(
                    target=name,
                    source=source_name,
                    shape=list(parameter.shape),
                    kind="parameter",
                )
            )
            transferred_elements += parameter.numel()
        else:
            initialized.append(dict(name=name, shape=list(parameter.shape)))
    for name, buffer in model.named_buffers():
        retained = previous_buffers.get(id(buffer))
        source_name = retained[0] if retained is not None else explicit.get(name)
        if source_name is not None:
            mapping.append(
                dict(
                    target=name,
                    source=source_name,
                    shape=list(buffer.shape),
                    kind="buffer",
                )
            )
    audit = dict(
        source_model=MODELS[identifier],
        transferred=mapping,
        newly_initialized_parameters=initialized,
        transferred_parameter_elements=transferred_elements,
        total_parameter_elements=sum(p.numel() for p in model.parameters()),
        new_modules_initialization="native baseline initializers",
    )
    model.transfer_audit = audit
    model.warm_revision = REVISION
    return model, audit


class WarmTrainer(PaperBaselineTrainer):
    baseline_id = "yolo11_dcrcf"

    def get_model(self, cfg=None, weights=None, verbose=True):
        if weights is None:
            raise ValueError("A stock GDUT Stage1 checkpoint is required.")
        validate_stock_model(weights, self.baseline_id)
        model = self.set_model_names_for_load(
            DetectionModel(
                cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=verbose
            )
        )
        model.load(weights)
        model, audit = transfer_baseline(model, self.baseline_id)
        source_state, target_state = weights.state_dict(), model.state_dict()
        for item in audit["transferred"]:
            target = target_state[item["target"]]
            expected = source_state[item["source"]].to(
                device=target.device, dtype=target.dtype
            )
            if not torch.equal(target, expected):
                raise ValueError(f"Transferred tensor differs: {item['target']}")
        report = dict(
            baseline_id=self.baseline_id,
            revision=REVISION,
            rank=os.environ.get("RANK", "-1"),
            modules=module_inventory(model),
            transfer=audit,
        )
        self.save_dir.mkdir(parents=True, exist_ok=True)
        (self.save_dir / f"architecture_rank_{report['rank']}.json").write_text(
            json.dumps(report, indent=2), encoding="utf-8"
        )
        return model

    def final_eval(self):
        super().final_eval()
        if RANK in {-1, 0}:
            path = self.save_dir / "training_complete.json"
            value = json.loads(path.read_text())
            value["revision"] = REVISION
            path.write_text(json.dumps(value, indent=2), encoding="utf-8")


class WarmDCRCFTrainer(WarmTrainer):
    baseline_id = "yolo11_dcrcf"


class WarmDWRTrainer(WarmTrainer):
    baseline_id = "yolov8_dwr_aspp_nwd"


TRAINERS = {"yolo11_dcrcf": WarmDCRCFTrainer, "yolov8_dwr_aspp_nwd": WarmDWRTrainer}


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_protocol.py
"""Stage1 and fine-tuning settings shared with the CLEAN A6 recipe."""

REVISION = "gdut_stage1_baselines_v1"
MODELS = {"yolo11_dcrcf": "yolo11n", "yolov8_dwr_aspp_nwd": "yolov8s"}
COMMON = {
    "epochs": 100,
    "batch": 16,
    "imgsz": 640,
    "seed": 3407,
    "optimizer": "AdamW",
    "lr0": 0.0001,
    "lrf": 0.01,
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "pretrained": True,
    "resume": False,
    "patience": 50,
    "deterministic": True,
    "workers": 4,
    "amp": True,
    "compile": False,
    "cos_lr": False,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.0,
    "nbs": 64,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.4,
    "bgr": 0.0,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "close_mosaic": 10,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "save": True,
    "save_period": -1,
    "plots": True,
    "cache": False,
    "val": True,
    "rect": False,
    "fraction": 1.0,
    "multi_scale": False,
}
EVALUATION = dict(
    imgsz=640,
    batch=16,
    conf=0.001,
    iou=0.7,
    max_det=300,
    workers=0,
    half=False,
    quantize=None,
    augment=False,
    plots=True,
)


def training_settings(device="0,1", stage1=False, smoke=False):
    values = dict(COMMON, device=device)
    if stage1:
        values["lr0"] = 0.0003
    if smoke:
        values.update(epochs=1, batch=2, imgsz=64, workers=0, plots=False, amp=False)
    return values


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_runner.py
"""Train stock nano Stage1 and fine-tune the baseline graphs from GDUT weights."""

import argparse
import gc
import json
from pathlib import Path
import random

import numpy as np
import torch
import ultralytics
from ultralytics import YOLO
import yaml

from baseline_model import assert_inventory, module_inventory
from gdut_data import prepared_dataset, sha256
from plain_logging import configure_plain_logging
from warm_data import dataset_yaml, find_stage1
from warm_environment import PIN, runtime_snapshot
from warm_metrics import export_metrics
from warm_model import TRAINERS
from warm_protocol import COMMON, EVALUATION, MODELS, REVISION, training_settings

configure_plain_logging()
CODE_DIR = Path(__file__).resolve().parent


def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )


def set_seed():
    random.seed(3407)
    np.random.seed(3407)
    torch.manual_seed(3407)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(3407)


def release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def check_version():
    if ultralytics.__version__ != PIN:
        raise RuntimeError(f"Use ultralytics=={PIN}.")


def validate_saved_args(path, configured):
    actual = yaml.safe_load(Path(path).read_text())
    for key in COMMON:
        if actual.get(key) != configured[key]:
            raise ValueError(f"Training setting changed: {key}.")
    return actual


def validation(model, data, root, device, smoke):
    settings = dict(EVALUATION)
    if smoke:
        settings.update(imgsz=64, batch=2, plots=False)
    result = model.val(
        data=data,
        split="val",
        device=device.split(",")[0],
        project=str(root),
        name="val_check",
        **settings,
    )
    values, classes, definitions = export_metrics(result)
    write_json(
        root / "validation_details.json",
        dict(metrics=values, per_class=classes, definitions=definitions),
    )
    return values


def train_stage1(
    prepared_dir="",
    device="0,1",
    working="/kaggle/working",
    expected_gpu="Tesla T4",
    smoke=False,
    source_model="yolo11n.pt",
):
    check_version()
    prepared, metadata, dataset_hash = prepared_dataset(prepared_dir)
    runtime = runtime_snapshot(device, expected_gpu, smoke)
    root = Path(working) / "stage1_runs/gdut_S1_yolo11n"
    if root.exists():
        raise FileExistsError(f"Use a fresh Stage1 directory: {root}")
    data = dataset_yaml(prepared, Path(working) / "stage1_config", False)
    args = training_settings(device, stage1=True, smoke=smoke)
    args.update(data=data, project=str(root.parent), name=root.name, exist_ok=False)
    set_seed()
    stock = YOLO(source_model)
    stock.train(**args)
    del stock
    release()
    best = root / "weights/best.pt"
    model = YOLO(str(best))
    actual_args = validate_saved_args(root / "args.yaml", args)
    metrics = validation(model, data, root, device, smoke)
    write_json(
        root / "run_manifest.json",
        dict(
            dataset="gdut",
            stage="S1",
            model="yolo11n",
            names=metadata["names"],
            imgsz=640,
            smoke=smoke,
            checkpoint="weights/best.pt",
            checkpoint_sha256=sha256(best),
            dataset_fingerprint=dataset_hash,
            source_pretrained_model="yolo11n.pt",
            training_args=actual_args,
            validation={
                "metrics/precision(B)": metrics["precision"],
                "metrics/recall(B)": metrics["recall"],
                "metrics/mAP50(B)": metrics["mAP50"],
                "metrics/mAP50-95(B)": metrics["mAP50_95"],
            },
            runtime=runtime,
            recipe="conservative_v2_nano_scale",
            initialization="stock_pretrained",
            warm_revision=REVISION,
        ),
    )
    print(f"Stock YOLO11n Stage1 complete: {root}", flush=True)
    return root


def train_baseline(
    identifier,
    prepared_dir="",
    checkpoint_dir="",
    device="0,1",
    working="/kaggle/working",
    expected_gpu="Tesla T4",
    smoke=False,
    input_root="/kaggle/input",
):
    check_version()
    prepared, metadata, dataset_hash = prepared_dataset(prepared_dir)
    weight, source = find_stage1(
        input_root, MODELS[identifier], dataset_hash, checkpoint_dir, smoke
    )
    runtime = runtime_snapshot(device, expected_gpu, smoke)
    root = Path(working) / "transfer_runs" / ("gdut_" + identifier)
    if root.exists():
        raise FileExistsError(f"Use a fresh fine-tuning directory: {root}")
    data = dataset_yaml(prepared, Path(working) / "transfer_config", True)
    args = training_settings(device, smoke=smoke)
    args.update(
        model=str(weight),
        data=data,
        project=str(root.parent),
        name=root.name,
        exist_ok=False,
    )
    set_seed()
    trainer = TRAINERS[identifier](overrides=args)
    trainer.train()
    del trainer
    release()
    marker = json.loads((root / "training_complete.json").read_text())
    if not 1 <= marker["epochs_completed"] <= args["epochs"]:
        raise ValueError("Invalid completed epoch count.")
    actual_args = validate_saved_args(root / "args.yaml", args)
    best = root / "weights/best.pt"
    model = YOLO(str(best))
    if model.model.warm_revision != REVISION or model.model.baseline_id != identifier:
        raise ValueError("Checkpoint and fine-tuning run disagree.")
    assert_inventory(model.model)
    modules, audit = module_inventory(model.model), model.model.transfer_audit
    parameters = sum(p.numel() for p in model.model.parameters())
    ranks = (
        [-1] if device == "cpu" or "," not in device else range(len(device.split(",")))
    )
    for rank in ranks:
        record = json.loads((root / f"architecture_rank_{rank}.json").read_text())
        if record["modules"] != modules:
            raise ValueError(
                "Distributed workers constructed different baseline graphs."
            )
    metrics = validation(model, data, root, device, smoke)
    write_json(root / "transfer_audit.json", audit)
    write_json(
        root / "run_manifest.json",
        dict(
            dataset="gdut",
            names=metadata["names"],
            baseline_id=identifier,
            warm_revision=REVISION,
            model=MODELS[identifier],
            imgsz=640,
            stage="baseline_finetune",
            smoke=smoke,
            checkpoint="weights/best.pt",
            checkpoint_sha256=sha256(best),
            dataset_fingerprint=dataset_hash,
            augmentation_manifest_sha256=sha256(
                prepared / "augmentation_manifest.json"
            ),
            source_checkpoint_sha256=source["checkpoint_sha256"],
            source_manifest=source,
            initialization="stock_GDUT_Stage1",
            hardaug=True,
            training_args=actual_args,
            epochs_completed=marker["epochs_completed"],
            best_epoch=marker.get("best_epoch"),
            validation=metrics,
            modules=modules,
            transfer=audit,
            parameters=parameters,
            runtime=runtime,
            code_sha256={p.name: sha256(p) for p in CODE_DIR.glob("*.py")},
        ),
    )
    del model
    release()
    print(f"Baseline fine-tuning complete: {root}", flush=True)
    return root


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--stage1-nano", action="store_true")
    parser.add_argument("--baseline", choices=MODELS)
    parser.add_argument("--prepared-dir", default="")
    parser.add_argument("--checkpoint-dir", default="")
    parser.add_argument("--device", default="0,1")
    parser.add_argument("--expected-gpu", default="Tesla T4")
    parser.add_argument("--smoke", action="store_true")
    args = parser.parse_args()
    if args.stage1_nano:
        train_stage1(
            args.prepared_dir,
            args.device,
            expected_gpu=args.expected_gpu,
            smoke=args.smoke,
        )
    elif args.baseline:
        train_baseline(
            args.baseline,
            args.prepared_dir,
            args.checkpoint_dir,
            args.device,
            expected_gpu=args.expected_gpu,
            smoke=args.smoke,
        )
    else:
        parser.error("Select --stage1-nano or --baseline.")


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/b1_lr_run.py
"""Run the approved B1 paired pilot on Kaggle using isolated DDP child processes."""

import argparse
from collections import Counter
import csv
import hashlib
import json
import math
import os
from pathlib import Path
import sys
from tempfile import mkdtemp

from PIL import Image
import torch
import yaml
from ultralytics import YOLO

from baseline_model import assert_inventory, module_inventory
from b1_lr_trainer import AMP_INITIAL_SCALE, PairedLRTrainer, arm_multiplier, image_inventory
from gdut_data import NAMES, prepared_dataset, sha256
from warm_data import dataset_yaml, find_stage1
from warm_environment import runtime_snapshot
from warm_launch import run_logged
from warm_protocol import REVISION, training_settings
from warm_runner import check_version, set_seed, validate_saved_args

DATA_SHA = "026a2afb617254b9dd2b9461651bce6f74295d18f29d52a790bf755d32944f65"
AUG_SHA = "95318f647ea1493b9d83779d150abf94ad4a6c32c54f3e3505a86f078d837342"
STAGE1_SHA = "ea174f78b6b768a2f436c4248a7b95e3001ca3ff47fe2d850a78c22f3e98a88c"
ARCHIVE_SHA = "52f4490280ed92e378d2d9936256cb286a1b2fd102114c0f226324b8935553d7"
DEVICE = "0,1"
SEED = 3407
ARMS = {"control": 1.0, "treatment": 3.0}


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path, data):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def unique(paths, message):
    if len(paths) != 1:
        raise RuntimeError(f"{message}. Found {len(paths)}")
    return paths[0]


def lock_inputs(input_root):
    data_path = unique(
        [p for p in input_root.rglob("dataset_manifest.json") if sha256(p) == DATA_SHA],
        "Mount the original prepared GDUT output used in A",
    )
    prepared, metadata, digest = prepared_dataset(str(data_path.parent))
    if digest != DATA_SHA or sha256(prepared / "augmentation_manifest.json") != AUG_SHA:
        raise ValueError("Preparation or augmentation manifest changed.")
    manifests = [(p, read_json(p)) for p in input_root.rglob("run_manifest.json")]
    stage1_source = unique(
        [p for p, m in manifests if m.get("checkpoint_sha256") == STAGE1_SHA],
        "Mount stock YOLO11n Stage1 version 363123845",
    )
    weight, source = find_stage1(input_root, "yolo11n", DATA_SHA, str(stage1_source.parent))
    if sha256(weight) != STAGE1_SHA:
        raise ValueError("Stage1 checkpoint hash changed.")
    historical_manifest = unique(
        [p for p, m in manifests if m.get("checkpoint_sha256") == ARCHIVE_SHA],
        "Mount Thanh's B1 output version 363134475 for historical ledger and module hashes",
    )
    historical = read_json(historical_manifest)
    archive = historical_manifest.parent / "weights/best.pt"
    if (
        historical["checkpoint"] != "weights/best.pt"
        or historical["source_checkpoint_sha256"] != STAGE1_SHA
        or historical["dataset_fingerprint"] != DATA_SHA
        or sha256(archive) != ARCHIVE_SHA
    ):
        raise ValueError("Historical B1 identity differs from A.")
    modules = historical["code_sha256"]
    code_dir = Path(__file__).resolve().parent
    if len(modules) != 17 or any(sha256(code_dir / name) != digest for name, digest in modules.items()):
        raise ValueError("A protected source module differs from the archived B1.")
    records = {split: sorted((r for r in metadata["records"] if r["split"] == split), key=lambda r: r["id"])
               for split in ("train", "val")}
    if len(records["train"]) != 1265 or len(records["val"]) != 316:
        raise ValueError("Expected 1265 base train and 316 validation images.")
    if sum(box[0] == 4 for row in records["val"] for box in row["boxes"]) != 404:
        raise ValueError("Expected 404 class none boxes in validation.")
    for split, rows in records.items():
        image_dir = prepared / "base/images" / split
        label_dir = prepared / "base/labels" / split
        ids = {row["id"] for row in rows}
        if len(ids) != len(rows) or {p.name for p in image_dir.iterdir()} != {f"{uid}.jpg" for uid in ids}:
            raise ValueError(f"Image IDs differ in {split}.")
        if {p.name for p in label_dir.iterdir()} != {f"{uid}.txt" for uid in ids}:
            raise ValueError(f"Label IDs differ in {split}.")
        for row in rows:
            uid = row["id"]
            with Image.open(image_dir / f"{uid}.jpg") as opened:
                rgb = opened.convert("RGB")
                digest = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
                if opened.getexif().get(274, 1) != 1 or digest != row["pixel_hash"]:
                    raise ValueError(f"Unexpected image pixels or orientation: {uid}")
            expected = Counter(tuple(round(float(v), 10) for v in box) for box in row["boxes"])
            actual = Counter(
                tuple(round(float(v), 10) for v in line.split())
                for line in (label_dir / f"{uid}.txt").read_text().splitlines() if line.strip()
            )
            if actual != expected:
                raise ValueError(f"Unexpected label: {uid}")
    hardaug_ids = {f"{row['id']}_hardaug" for row in records["train"]}
    hardaug_digest = hashlib.sha256()
    for kind, extension in (("images", "jpg"), ("labels", "txt")):
        entries = sorted((prepared / "hardaug" / kind / "train").iterdir())
        if {p.name for p in entries} != {f"{uid}.{extension}" for uid in hardaug_ids}:
            raise ValueError(f"Unexpected hardaug {kind} inventory.")
        if any(not p.is_file() for p in entries):
            raise ValueError(f"Hardaug {kind} must all be files.")
        for path in entries:
            hardaug_digest.update(f"{kind}/{path.name}:{sha256(path)}\n".encode())
    return prepared, weight, source, historical, records, hardaug_digest.hexdigest()


def build_configs(prepared, records, root):
    # Reuse the original hard-augmentation inventory checks and native full loader.
    original = Path(dataset_yaml(prepared, root, augmented=True))
    data = yaml.safe_load(original.read_text())
    data.pop("test", None)
    full = root / "full.yaml"
    full.write_text(yaml.safe_dump(data))
    selected = sorted(records["train"], key=lambda r: hashlib.sha256(
        f"{SEED}:{r['id']}".encode()
    ).digest())[:64]
    train_paths = [
        prepared / relative / filename
        for row in selected
        for relative, filename in (
            ("base/images/train", f"{row['id']}.jpg"),
            ("hardaug/images/train", f"{row['id']}_hardaug.jpg"),
        )
    ]
    smoke_paths = root / "smoke_train.txt"
    smoke_paths.write_text("".join(f"{p}\n" for p in train_paths))
    smoke = root / "smoke.yaml"
    smoke.write_text(yaml.safe_dump({**data, "train": str(smoke_paths)}))
    return {"smoke": smoke, "full": full}, {
        "smoke": image_inventory(train_paths),
        "full": image_inventory(
            image for directory in data["train"] for image in Path(directory).glob("*.jpg")
        ),
        "val": image_inventory(Path(data["val"]) / f"{row['id']}.jpg" for row in records["val"]),
    }


def train_child(weight, data, project, name, phase):
    check_version()
    options = training_settings(DEVICE, smoke=False)
    if phase == "smoke":
        # Two epochs allow a warmup epoch and a native scheduler epoch.
        options.update(epochs=2, workers=0, plots=False, close_mosaic=0)
    options.update(model=str(weight), data=str(data), project=str(project), name=name, exist_ok=False)
    set_seed()
    trainer = PairedLRTrainer(overrides=options)
    trainer.train()
    validate_saved_args(project / name / "args.yaml", options)


def arm_parameter_contract(group_report):
    return {key: (group["kind"], group["weight_decay"], group["betas"], group["lr"],
                  key in group["frozen"])
            for group in group_report["groups"] for key in group["names"]}


def validate_training_marker(completed, phase):
    epochs = completed.get("epochs_completed")
    if type(epochs) is not int or not 1 <= epochs <= 100 or (phase == "smoke" and epochs != 2):
        raise ValueError(f"Unexpected {phase} training length.")
    best_epoch = completed.get("best_epoch")
    # This pinned trainer calls EarlyStopping with epoch + 1.
    if completed.get("revision") != REVISION or type(best_epoch) is not int or not 1 <= best_epoch <= epochs:
        raise ValueError("Expected a one-based early stopping epoch within the completed run.")
    return epochs


def verify_arm(path, phase, multiplier, historical, inventory):
    completed = read_json(path / "training_complete.json")
    epochs = validate_training_marker(completed, phase)
    with (path / "results.csv").open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    if len(rows) != epochs or any(not math.isfinite(float(v)) for row in rows for v in row.values()):
        raise ValueError("Incomplete or nonfinite training CSV.")
    best = path / "weights/best.pt"
    model = YOLO(str(best)).model
    assert_inventory(model)
    if any(not torch.isfinite(tensor).all() for tensor in model.state_dict().values() if tensor.is_floating_point()):
        raise ValueError(f"Nonfinite best checkpoint: {best}")
    if module_inventory(model) != historical["modules"] or model.transfer_audit != historical["transfer"]:
        raise ValueError("Checkpoint architecture or Stage1 ledger differs from A.")
    expected_total = historical["parameters"]
    if sum(p.numel() for p in model.parameters()) != expected_total:
        raise ValueError("B1 parameter count changed.")
    reports = []
    traces = []
    epoch_records = []
    expected_new = {item["name"] for item in historical["transfer"]["newly_initialized_parameters"]}
    for rank in (0, 1):
        head_shapes = read_json(path / f"detect_input_rank_{rank}.json")
        if [shape[-2:] for shape in head_shapes] != [[80, 80], [40, 40], [20, 20]]:
            raise ValueError(f"Invalid Detect input evidence on rank {rank}.")
        architecture = read_json(path / f"architecture_rank_{rank}.json")
        report = read_json(path / f"optimizer_rank_{rank}.json")
        if architecture["transfer"] != historical["transfer"] or architecture["modules"] != historical["modules"]:
            raise ValueError(f"Unexpected transfer or graph on rank {rank}.")
        if report["rank"] != rank or not report["amp"] or report["device"] != f"cuda:{rank}":
            raise ValueError(f"DDP/AMP mismatch on rank {rank}.")
        if report["multiplier"] != multiplier:
            raise ValueError(f"Unexpected arm multiplier on rank {rank}.")
        if report["train_inventory"] != inventory[phase] or report["val_inventory"] != inventory["val"]:
            raise ValueError(f"Dataloader retained a different train or val inventory on rank {rank}.")
        rank_epochs = [json.loads(line) for line in (path / f"training_epochs_rank_{rank}.jsonl").read_text().splitlines()]
        if [row["epoch"] for row in rank_epochs] != list(range(1, epochs + 1)):
            raise ValueError(f"Missing or repeated training epoch on rank {rank}.")
        if any(not 1 <= row["successful_steps"] <= row["attempted_steps"] for row in rank_epochs):
            raise ValueError(f"Missing successful optimizer updates on rank {rank}: {rank_epochs}")
        parameter_contract = arm_parameter_contract(report)
        if len(parameter_contract) != sum(len(g["names"]) for g in report["groups"]):
            raise ValueError("Invalid optimizer receipt duplicates.")
        for name, (kind, decay, betas, lr, frozen) in parameter_contract.items():
            expected_lr = 1e-4 * (multiplier if name in expected_new and not frozen else 1.0)
            if lr != expected_lr or kind not in ("weight", "bn", "bias") or betas != [0.9, 0.999]:
                raise ValueError(f"Incorrect optimizer settings for {name}")
            if not math.isclose(decay, 0.0005 if kind == "weight" else 0.0):
                raise ValueError(f"Incorrect decay for {name}")
        trace = [json.loads(line) for line in (path / f"optimizer_steps_rank_{rank}.jsonl").read_text().splitlines()]
        # Evidence of positive finite gradients and updates for DCNv2 and the three head inputs.
        for index in (6, 8, 14, 16, 18, 19):
            if not any(
                row["grad_rms"] > 0 and row["update_rms"] > 0
                for step in trace for key, row in step["parameters"].items() if key.startswith(f"model.{index}.")
            ):
                raise ValueError(f"No measured positive gradient/update in layer {index} on rank {rank}.")
        ranks_by_epoch = {event["epoch"] for event in trace}
        if ranks_by_epoch != set(range(1, epochs + 1)):
            raise ValueError(f"Optimizer steps missing for a training epoch on rank {rank}.")
        reports.append(report)
        traces.append(trace)
        epoch_records.append(rank_epochs)
    if reports[0]["initial_sha256"] != reports[1]["initial_sha256"]:
        raise ValueError("DDP ranks did not start from synchronized model state.")
    if reports[0]["groups"] != reports[1]["groups"]:
        raise ValueError("DDP ranks disagree on the optimizer groups.")
    return best, completed, reports, traces, epoch_records


def verify_pair(left, right):
    for a, b in zip(left[2], right[2], strict=True):
        if a["initial_sha256"] != b["initial_sha256"]:
            raise ValueError("The two arms did not start from the same per-rank state.")
        original, treatment = arm_parameter_contract(a), arm_parameter_contract(b)
        if original.keys() != treatment.keys():
            raise ValueError("The arms optimized a different parameter inventory.")
        for name, fields in original.items():
            if fields[:3] != treatment[name][:3] or fields[4] != treatment[name][4]:
                raise ValueError(f"Arm settings other than LR differ for {name}.")
    for arm in (left, right):
        for report, rank_trace in zip(arm[2], arm[3], strict=True):
            base = [group["lr"] for group in report["groups"]]
            for point in rank_trace:
                ratios = [lr / initial for lr, initial in zip(point["lr"], base, strict=True)]
                if not all(math.isfinite(r) and 0 <= r <= 1.000001 for r in ratios):
                    raise ValueError("Invalid scheduler or warmup LR.")
                if max(ratios) - min(ratios) > 1e-6:
                    raise ValueError("Scheduler changed the intended per-group LR ratio.")
    for rank, (a, b) in enumerate(zip(left[4], right[4], strict=True)):
        if any(x["batch_order_sha256"] != y["batch_order_sha256"] for x, y in zip(a, b)):
            raise ValueError(f"The arms saw different training batch order on rank {rank}.")


def run(mode, input_root, output_base, run_root=None):
    from b1_lr_evaluate import evaluate_pair

    check_version()
    runtime = runtime_snapshot(DEVICE, "Tesla T4")
    if not os.environ.get("CUBLAS_WORKSPACE_CONFIG"):
        raise RuntimeError("Apply the original runtime cell before launch.")
    prepared, weight, source, historical, records, hardaug_digest = lock_inputs(input_root)
    if mode == "evaluate_full":
        root = run_root.resolve(strict=True)
        if (root / "PAIR_EVALUATED.json").exists():
            raise ValueError("This full pair already has an evaluation receipt.")
        launch = read_json(root / "launch_manifest.json")
        trained = read_json(root / "FULL_TRAINED.json")
        data = root / "full.yaml"
        if (trained["launch_manifest_sha256"] != sha256(root / "launch_manifest.json")
                or launch["hardaug_content_sha256"] != hardaug_digest
                or launch["configs"]["full"] != sha256(data)):
            raise ValueError("Saved full-pair inputs differ from the completed training evidence.")
        configuration = yaml.safe_load(data.read_text())
        if configuration.get("val") != str(prepared / "base/images/val") or "test" in configuration:
            raise ValueError("The saved full configuration needs the same validation mount and no test split.")
        evidence = {}
        for arm, multiplier in ARMS.items():
            arm_dir = root / "full" / arm
            if sha256(arm_dir / "weights/best.pt") != trained["checkpoints_sha256"][arm]:
                raise ValueError(f"The {arm} checkpoint differs from the completed training receipt.")
            evidence[arm] = verify_arm(arm_dir, "full", multiplier, historical, launch["expected_loader_inventory"])
        verify_pair(evidence["control"], evidence["treatment"])
        evaluate_pair(root, data, prepared, records["val"], evidence, phase="full")
        print(f"B1 evaluation recovered from {root}", flush=True)
        return
    output_base.mkdir(parents=True, exist_ok=True)
    root = Path(mkdtemp(prefix="seed-3407-", dir=output_base))
    configs, inventory = build_configs(prepared, records, root)
    write_json(root / "launch_manifest.json", {
        "mode": mode, "seed": SEED, "runtime": runtime, "dataset_sha256": DATA_SHA,
        "amp_initial_scale": AMP_INITIAL_SCALE,
        "stage1_sha256": source["checkpoint_sha256"], "augmentation_sha256": AUG_SHA,
        "hardaug_content_sha256": hardaug_digest,
        "source_stage2_sha256": ARCHIVE_SHA, "configs": {p: sha256(c) for p, c in configs.items()},
        "expected_loader_inventory": inventory,
        "review_code_sha256": {p.name: sha256(p) for p in Path(__file__).parent.glob("b1_lr_*.py")},
    })
    for phase in (("smoke",) if mode == "smoke" else ("smoke", "full")):
        project = root / phase
        project.mkdir()
        evidence = {}
        for arm, multiplier in ARMS.items():
            command = [
                sys.executable, str(Path(__file__).resolve()), "child",
                "--weight", str(weight), "--data", str(configs[phase]),
                "--project", str(project), "--arm", arm, "--phase", phase,
            ]
            os.environ["B1_LR_MULTIPLIER"] = str(multiplier)
            try:
                run_logged(command, root / f"{phase}_{arm}.log")
            finally:
                os.environ.pop("B1_LR_MULTIPLIER", None)
            evidence[arm] = verify_arm(project / arm, phase, multiplier, historical, inventory)
        verify_pair(evidence["control"], evidence["treatment"])
        checkpoints = {arm: sha256(info[0]) for arm, info in evidence.items()}
        if phase == "smoke":
            evaluate_pair(root, configs[phase], prepared, records["val"], evidence, phase=phase)
            write_json(root / "SMOKE_READY.json", {
                "seed": SEED, "rank_count": 2, "mode": "real_data_smoke",
                "checkpoints_sha256": checkpoints,
                "evaluator_receipt_sha256": sha256(root / "smoke_PAIR_EVALUATED.json"),
            })
        else:
            write_json(root / "FULL_TRAINED.json", {
                "launch_manifest_sha256": sha256(root / "launch_manifest.json"),
                "checkpoints_sha256": checkpoints,
            })
            evaluate_pair(root, configs[phase], prepared, records["val"], evidence, phase=phase)
    print(f"B1 evidence saved to {root}", flush=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("mode", choices=("smoke", "smoke_then_pair", "evaluate_full", "child"))
    parser.add_argument("--input-root", type=Path, default=Path("/kaggle/input"))
    parser.add_argument("--output-base", type=Path, default=Path("/kaggle/working/b1_lr_pilot"))
    parser.add_argument("--run-root", type=Path)
    parser.add_argument("--weight", type=Path)
    parser.add_argument("--data", type=Path)
    parser.add_argument("--project", type=Path)
    parser.add_argument("--arm", choices=ARMS)
    parser.add_argument("--phase", choices=("smoke", "full"))
    args = parser.parse_args()
    if args.mode == "child":
        if not all((args.weight, args.data, args.project, args.arm, args.phase)):
            parser.error("Child mode requires weight, data, project, arm and phase.")
        if arm_multiplier() != ARMS[args.arm]:
            raise ValueError("Child arm disagrees with the inherited LR multiplier.")
        train_child(args.weight, args.data, args.project, args.arm, args.phase)
    else:
        if (args.mode == "evaluate_full") != (args.run_root is not None):
            parser.error("Only evaluate_full requires --run-root.")
        run(args.mode, args.input_root, args.output_base, args.run_root)


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/b1_lr_evaluate.py
"""Lượt A diagnostic replay, shared by both pilot arms under one metric contract."""

import csv
from collections import Counter
import json
import math
from pathlib import Path
from tempfile import mkdtemp

import cv2
import torch
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionValidator
from ultralytics.utils.nms import non_max_suppression
from ultralytics.utils.ops import xywh2xyxy

from gdut_data import NAMES, sha256
from warm_metrics import export_metrics
from warm_protocol import EVALUATION

PRECISION_FLOOR_TP = 306
PRECISION_FLOOR_PREDICTIONS = 367
PRECISION_FLOOR = PRECISION_FLOOR_TP / PRECISION_FLOOR_PREDICTIONS
MAX_AP_DROP = 0.001  # 0.10 percentage points.
ARCHIVED_A = {
    "native_reference_kaggle_version": 363134475,
    "native_reference": {"mAP50": 0.8885655148067866, "mAP50_95": 0.5747310221861304},
    "diagnostic_kaggle_version": 372212552,
    "diagnostic_fixed_025": {"threshold": 0.25, "tp": 306, "fp": 61, "fn": 98},
}

THRESHOLDS = (0.10, 0.15, 0.20, 0.25, 0.30, 0.35)
SHAPES = {"P3": (80, 80), "P4": (40, 40), "P5": (20, 20)}
TRACE_LAYERS = (6, 8, 11, 12, 13, 14, 15, 16, 17, 18, 19)


def write_json(path, value):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def write_csv(path, rows, fields=None):
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields or list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def box_iou(box, boxes):
    start = torch.maximum(box[:2], boxes[:, :2])
    end = torch.minimum(box[2:], boxes[:, 2:])
    overlap = (end - start).clamp_min(0).prod(1)
    total = (box[2:] - box[:2]).prod() + (boxes[:, 2:] - boxes[:, :2]).prod(1)
    return overlap / (total - overlap).clamp_min(1e-8)


def match_targets(detections, targets):
    # Deterministic tie-break uses the original detection and annotation order.
    candidates = torch.where(detections[:, 5] == 4)[0]
    ordered = sorted(candidates.tolist(), key=lambda i: (-float(detections[i, 4]), i))
    unmatched = list(range(len(targets)))
    pairs = []
    for index in ordered:
        if unmatched:
            ious = box_iou(detections[index, :4], targets[unmatched])
            best = int(ious.argmax())
            if ious[best] >= 0.50:
                pairs.append((index, unmatched.pop(best)))
    return ordered, pairs


def image_and_targets(record, image_dir, device):
    image = cv2.imread(str(image_dir / f"{record['id']}.jpg"))
    height, width = image.shape[:2]
    ratio = min(640 / height, 640 / width)
    new_width, new_height = round(width * ratio), round(height * ratio)
    left, top = round((640 - new_width) / 2 - 0.1), round((640 - new_height) / 2 - 0.1)
    if (new_width, new_height) != (width, height):
        image = cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_LINEAR)
    canvas = cv2.copyMakeBorder(
        image, top, 640 - new_height - top, left, 640 - new_width - left,
        cv2.BORDER_CONSTANT, value=(114, 114, 114),
    )
    image = torch.from_numpy(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)).permute(2, 0, 1)
    targets = []
    for cls, x, y, w, h in record["boxes"]:
        if cls == 4:
            targets.append([
                (x - w / 2) * width * ratio + left, (y - h / 2) * height * ratio + top,
                (x + w / 2) * width * ratio + left, (y + h / 2) * height * ratio + top,
            ])
    return image[None].to(device).float() / 255, torch.tensor(targets).reshape(-1, 4)


def case_details(uid, targets, raw_boxes, scores, detections):
    _, pairs = match_targets(detections, targets)
    matched = {target for _, target in pairs}
    winners = scores.argmax(dim=0)
    max_score = scores.max(dim=0).values
    output = []
    for index, target in enumerate(targets):
        ious = box_iou(target, raw_boxes)
        overlap = ious >= 0.50
        none_qualifies = overlap & (scores[4] >= 0.25)
        same_class = none_qualifies & (winners == 4)
        other_class_qualifies = overlap & (winners != 4) & (max_score >= 0.25)
        if index in matched:
            reason = "matched"
        elif not overlap.any():
            reason = "no_raw_box_iou50"
        elif not none_qualifies.any():
            reason = "raw_none_below_confidence"
        elif not same_class.any():
            reason = "raw_overlap_other_class"
        else:
            duplicate = (box_iou(target, detections[:, :4]) >= 0.50) & (detections[:, 5] == 4)
            reason = "one_to_one_conflict" if duplicate.any() else "lost_after_postprocessing"
        peak = int(torch.where(overlap, scores[4], -1).argmax()) if overlap.any() else None
        level = None if peak is None else ("P3" if peak < 6400 else "P4" if peak < 8000 else "P5")
        output.append({
            "image_id": uid, "target_index": index, "result": reason,
            "sqrt_area_input_px": float((target[2:] - target[:2]).prod().sqrt()),
            "max_raw_iou": float(ious.max()), "best_none_conf_at_iou50": None if peak is None else float(scores[4, peak]),
            "winning_class_at_peak": None if peak is None else NAMES[int(winners[peak])],
            "pyramid_level": level, "other_class_above_confidence": bool(other_class_qualifies.any()),
        })
    return output


def trace_first_image(network, image):
    traces = {}

    def capture(index):
        def observe(module, inputs, output):
            tensor = output[0] if isinstance(output, (tuple, list)) else output
            if not isinstance(tensor, torch.Tensor) or not torch.isfinite(tensor).all():
                raise ValueError(f"Invalid tensor in layer {index}.")
            traces[index] = {"type": type(module).__name__, "shape": list(tensor.shape),
                             "rms": float(tensor.float().square().mean().sqrt())}
            if index == 19:
                shapes = [tuple(feature.shape[-2:]) for feature in inputs[0]]
                if shapes != list(SHAPES.values()):
                    raise ValueError(f"Unexpected Detect shapes: {shapes}")
                traces[index]["input_shapes"] = shapes
        return observe

    hooks = [network.model[index].register_forward_hook(capture(index)) for index in TRACE_LAYERS]
    try:
        raw = network(image)
    finally:
        for hook in hooks:
            hook.remove()
    return raw, traces


def operating_point(curve):
    # Cross multiplication keeps the 306/367 boundary exact.
    return max(
        (
            point for point in curve
            if point["tp"] > 0 and point["tp"] * PRECISION_FLOOR_PREDICTIONS
            >= (point["tp"] + point["fp"]) * PRECISION_FLOOR_TP
        ),
        key=lambda point: (point["tp"], point["precision"], point["threshold"]),
        default=None,
    )


def replay(net, records, image_dir, device, OUTPUT):
    """Retain the original square 640 / class-argmax / IoU .50 replay."""
    EXPECTED_IMAGES = len(records)
    EXPECTED_NONE = sum(box[0] == 4 for row in records for box in row["boxes"])
    network = net.to(device).float().eval()
    error_rows = []
    image_predictions = []
    all_ranked = []
    counts = {threshold: Counter(tp=0, fp=0, fn=0) for threshold in THRESHOLDS}
    traces = {}
    with torch.inference_mode():
        for index, record in enumerate(records):
            image, targets = image_and_targets(record, image_dir, device)
            if index == 0:
                prediction, traces = trace_first_image(network, image)
            else:
                prediction = network(image)
            raw = prediction[0] if isinstance(prediction, (tuple, list)) else prediction
            if list(raw.shape) != [1, 9, 8400] or not torch.isfinite(raw).all():
                raise ValueError(f"Invalid decoded B1 tensor on image {record['id']}.")
            detections = non_max_suppression(
                raw.clone(), conf_thres=0.001, iou_thres=0.70, nc=5, max_det=300, multi_label=False,
            )[0].float().cpu()
            boxes = xywh2xyxy(raw[0, :4].T).float().cpu()
            scores = raw[0, 4:].float().cpu()
            selected = detections[detections[:, 4] >= 0.25]
            error_rows.extend(case_details(record["id"], targets, boxes, scores, selected))
            ordered, pairs = match_targets(detections, targets)
            true_detections = {detection for detection, _ in pairs}
            all_ranked.extend((float(detections[j, 4]), j in true_detections) for j in ordered)
            for confidence in THRESHOLDS:
                kept = [j for j in ordered if float(detections[j, 4]) >= confidence]
                tp = sum(j in true_detections for j in kept)
                counts[confidence].update(tp=tp, fp=len(kept)-tp, fn=len(targets)-tp)
            image_predictions.append({
                "image_id": record["id"], "detections_xyxy_conf_class": detections[:, :6].tolist(),
            })
            if (index + 1) % 64 == 0:
                print(f"Diagnostic replay: {index + 1}/{len(records)} images", flush=True)

    if len(error_rows) != EXPECTED_NONE or len(image_predictions) != EXPECTED_IMAGES:
        raise ValueError("Incomplete class none or image coverage.")
    pr_rows = []
    for threshold, count in counts.items():
        tp, fp, fn = count["tp"], count["fp"], count["fn"]
        if tp + fn != EXPECTED_NONE:
            raise ValueError(f"Inconsistent TP/FN at threshold {threshold:.2f}.")
        pr_rows.append({"threshold": threshold, "tp": tp, "fp": fp, "fn": fn,
                        "precision": tp / (tp + fp) if tp + fp else 0.0, "recall": tp / EXPECTED_NONE})
    curve = []
    tp = fp = 0
    for score, correct in sorted(all_ranked, key=lambda row: -row[0]):
        tp += int(correct)
        fp += int(not correct)
        point = {"threshold": score, "precision": tp / (tp + fp), "recall": tp / EXPECTED_NONE,
                 "tp": tp, "fp": fp, "fn": EXPECTED_NONE - tp}
        if curve and curve[-1]["threshold"] == score:
            curve[-1] = point
        else:
            curve.append(point)
    write_csv(OUTPUT / "none_pr.csv", pr_rows)
    write_csv(OUTPUT / "none_pr_curve.csv", curve, ("threshold", "precision", "recall", "tp", "fp", "fn"))
    write_csv(OUTPUT / "none_error_breakdown.csv", error_rows)
    write_json(OUTPUT / "diagnostic_predictions.json", image_predictions)
    write_json(OUTPUT / "module_probe.json", {"image_id": records[0]["id"], "layers": traces})
    fixed = next(row for row in pr_rows if row["threshold"] == 0.25)
    if fixed["tp"] != sum(row["result"] == "matched" for row in error_rows):
        raise ValueError("Case trace disagrees with TP at confidence 0.25.")
    by_size_and_pyramid = Counter(
        ("small" if row["sqrt_area_input_px"] < 32 else "other",
         row["pyramid_level"] or "no_iou50", row["result"])
        for row in error_rows
    )
    return {
        "fixed_025": fixed,
        "at_precision_floor": operating_point(curve),
        "by_size_pyramid_reason": [
            {"size": size, "pyramid": pyramid, "reason": reason, "count": count}
            for (size, pyramid, reason), count in sorted(by_size_and_pyramid.items())
        ],
    }


def native_validation(best, data, records, output, device):
    seen = []

    class AuditedValidator(DetectionValidator):
        def update_metrics(self, predictions, batch):
            seen.extend(Path(path).stem for path in batch["im_file"])
            super().update_metrics(predictions, batch)

    model = YOLO(str(best))
    metrics = model.val(validator=AuditedValidator, data=str(data), split="val",
                        device=device, project=str(output), name="native_val",
                        save_json=True, **EVALUATION)
    values, classes, definitions = export_metrics(metrics)
    support = Counter(int(box[0]) for row in records for box in row["boxes"])
    if sorted(seen) != sorted(row["id"] for row in records):
        raise ValueError("Native validator saw a different image inventory.")
    if metrics.nt_per_class.tolist() != [support[index] for index in range(len(NAMES))]:
        raise ValueError("Native validator saw a different class support.")
    if not all(math.isfinite(values[key]) for key in ("precision", "recall", "mAP50", "mAP50_95")):
        raise ValueError("Nonfinite native validation metrics.")
    for row in classes:
        row["support"] = support[row["class_id"]]
    write_csv(output / "per_class.csv", classes)
    write_json(output / "native_metrics.json", {
        "metrics": values, "definitions": definitions, "validation_images": len(seen),
        "checkpoint_sha256": sha256(best), "support": dict(support),
    })
    return values


def compare_arms(left, right, delta_ap):
    if left is None and right is None:
        return None, None, "precision_floor_not_met_by_both_arms"
    if left is None:
        return None, None, "precision_floor_not_met_by_control"
    if right is None:
        return None, None, "precision_floor_not_met_by_treatment"
    delta_tp = right["tp"] - left["tp"]
    delta_recall = right["recall"] - left["recall"]
    ap_within_limit = delta_ap >= -MAX_AP_DROP or math.isclose(delta_ap, -MAX_AP_DROP, rel_tol=0, abs_tol=1e-12)
    verdict = "signal_for_more_seeds" if delta_tp > 0 and ap_within_limit else "no_signal_under_pilot_criteria"
    return delta_tp, delta_recall, verdict


def evaluate_pair(root, data, prepared, records, evidence, phase, device="0"):
    prefix = "" if phase == "full" else "smoke_"
    attempt = Path(mkdtemp(prefix="evaluation-", dir=root / phase))
    summary = root / f"{prefix}paired_summary.json"
    receipt = root / f"{prefix}PAIR_EVALUATED.json"
    arms = {}
    for arm, (best, completed, _groups, _traces, epochs) in evidence.items():
        output = attempt / arm
        output.mkdir()
        net = YOLO(str(best)).model
        diagnostic = replay(net, records, prepared / "base/images/val", "cpu" if device == "cpu" else "cuda:0", output)
        del net
        metrics = native_validation(best, data, records, output, device)
        arms[arm] = {
            "checkpoint": str(best.relative_to(root)), "checkpoint_sha256": sha256(best),
            "epochs_completed": completed["epochs_completed"],
            "early_stopping_best_epoch": completed["best_epoch"],
            "successful_steps_by_rank": [sum(row["successful_steps"] for row in rank) for rank in epochs],
            "native_metrics": metrics, "diagnostic": diagnostic,
        }
    control, treatment = arms["control"], arms["treatment"]
    left = control["diagnostic"]["at_precision_floor"]
    right = treatment["diagnostic"]["at_precision_floor"]
    delta_ap = treatment["native_metrics"]["mAP50_95"] - control["native_metrics"]["mAP50_95"]
    delta_tp, delta_recall, verdict = compare_arms(left, right, delta_ap)
    if phase == "smoke":
        verdict = "smoke_evaluator_passed"
    write_json(summary, {
        "seed": 3407, "split": "val", "arms": arms, "precision_floor": PRECISION_FLOOR,
        "precision_floor_ratio": [PRECISION_FLOOR_TP, PRECISION_FLOOR_PREDICTIONS],
        "max_map50_95_drop": MAX_AP_DROP, "delta_none_recall": delta_recall,
        "delta_none_tp": delta_tp, "delta_map50_95": delta_ap, "verdict": verdict,
        "phase": phase, "epochs_equal": control["epochs_completed"] == treatment["epochs_completed"],
        "archived_A_reference": ARCHIVED_A,
        "diagnostic_convention": "square 640, argmax class, conf floor .001, NMS IoU .7, match IoU .50",
    })
    write_json(receipt, {
        "seed": 3407, "split": "val", "phase": phase, "verdict": verdict,
        "paired_summary_sha256": sha256(summary),
        "artifacts_sha256": {str(path.relative_to(root)): sha256(path)
                             for path in sorted(attempt.rglob("*")) if path.is_file()},
    })


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/b1_lr_trainer.py
"""Apply a differential learning rate to intact Ultralytics AdamW groups."""

import hashlib
import json
import math
import os
from pathlib import Path

import torch
from ultralytics.utils.torch_utils import unwrap_model
from warm_model import WarmDCRCFTrainer


AMP_INITIAL_SCALE = 16.0


def arm_multiplier():
    value = os.environ.get("B1_LR_MULTIPLIER")
    if value not in {"1.0", "3.0"}:
        raise ValueError("Launch through b1_lr_run.py with B1_LR_MULTIPLIER=1.0 or 3.0.")
    return float(value)


def partition_optimizer(optimizer, model, multiplier):
    parameters = dict(model.named_parameters())
    audit = model.transfer_audit
    new = {item["name"]: tuple(item["shape"]) for item in audit["newly_initialized_parameters"]}
    inherited = {
        item["target"]: tuple(item["shape"]) for item in audit["transferred"]
        if item["kind"] == "parameter"
    }
    if new.keys() & inherited.keys() or new | inherited != {
        name: tuple(value.shape) for name, value in parameters.items()
    }:
        raise ValueError("Transfer ledger must partition every named parameter by name and shape.")
    original = [param for group in optimizer.param_groups for param in group["params"]]
    original_ids = [id(param) for param in original]
    if len(original_ids) != len(set(original_ids)) or set(original_ids) != {id(p) for p in parameters.values()}:
        raise ValueError("Original AdamW groups must cover each parameter exactly once.")
    # Ultralytics owns frozen entries and decay/bias/BN classification.
    fresh = {
        id(parameters[name]) for name in new if parameters[name].requires_grad
    }
    if not fresh or not fresh < {id(p) for p in original if p.requires_grad}:
        raise ValueError("Both treatment and inherited trainable groups must be present.")
    if optimizer.state:
        raise ValueError("Split only a newly built optimizer, before its scheduler or first update.")

    if multiplier != 1.0:
        for group in list(optimizer.param_groups):
            plain = [param for param in group["params"] if id(param) not in fresh]
            boosted = [param for param in group["params"] if id(param) in fresh]
            if not boosted:
                continue
            if plain:
                group["params"] = plain
                optimizer.add_param_group({
                    **group, "params": boosted, "lr": group["lr"] * multiplier,
                })
            else:
                group["lr"] *= multiplier

    assigned = [id(p) for group in optimizer.param_groups for p in group["params"]]
    if sorted(assigned) != sorted(original_ids):
        raise ValueError("Optimizer partition lost or duplicated parameters.")
    names = {id(param): name for name, param in parameters.items()}
    return [
        {
            "kind": group["param_group"],
            "lr": group["lr"],
            "weight_decay": group["weight_decay"],
            "betas": list(group["betas"]),
            "names": [names[id(p)] for p in group["params"]],
            "frozen": [names[id(p)] for p in group["params"] if not p.requires_grad],
        }
        for group in optimizer.param_groups
    ]


def digest_initial_state(model):
    digest = hashlib.sha256()
    for name, value in model.state_dict().items():
        value = value.detach().cpu().contiguous()
        digest.update(f"{name}|{tuple(value.shape)}|{value.dtype}\n".encode())
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()


def image_inventory(paths):
    files = sorted(str(Path(path).resolve()) for path in paths)
    digest = hashlib.sha256("\n".join(files).encode()).hexdigest()
    return {"images": len(files), "paths_sha256": digest}


class PairedLRTrainer(WarmDCRCFTrainer):
    def _setup_train(self):
        super()._setup_train()
        # Both arms start from stock Stage1 and new DCRCF weights. Short smoke runs
        # can exhaust their updates before FP16 backs off from the default scale.
        self.scaler = torch.amp.GradScaler("cuda", init_scale=AMP_INITIAL_SCALE, enabled=self.amp)
        self.optimizer_attempts = 0
        # Register after ModelEMA has copied the model. Local hooks must never enter a checkpoint.
        detect = unwrap_model(self.model).model[19]
        rank = int(os.environ["RANK"])

        def verify_head_input(module, inputs):
            if not module.training:
                return
            features = inputs[0]
            shapes = [list(x.shape) for x in features]
            if [tuple(x.shape[-2:]) for x in features] != [(80, 80), (40, 40), (20, 20)]:
                raise ValueError(f"Unexpected B1 head input shapes: {shapes}")
            if not all(torch.isfinite(x).all() for x in features):
                raise ValueError("Nonfinite Detect input feature.")
            (self.save_dir / f"detect_input_rank_{rank}.json").write_text(json.dumps(shapes))
            handle.remove()

        handle = detect.register_forward_pre_hook(verify_head_input)

    def optimizer_step(self):
        self.optimizer_attempts += 1
        super().optimizer_step()

    def preprocess_batch(self, batch):
        self.batch_order.update(json.dumps(list(batch["im_file"])).encode() + b"\n")
        return super().preprocess_batch(batch)

    def build_optimizer(self, model, name="auto", lr=0.001, momentum=0.9, decay=1e-5, iterations=1e5):
        multiplier = arm_multiplier()
        if name != "AdamW" or lr != 1e-4:
            raise ValueError("Expected the original B1 AdamW and base LR 1e-4.")
        rank = int(os.environ.get("RANK", "-1"))
        if rank not in (0, 1) or str(self.device) != f"cuda:{rank}" or not self.amp:
            raise ValueError(f"B1 requires AMP on dual CUDA ranks; got rank={rank}, device={self.device}, amp={self.amp}.")
        base = super().build_optimizer(model, name, lr, momentum, decay, iterations)
        network = unwrap_model(model)
        groups = partition_optimizer(base, network, multiplier)
        report = {
            "rank": rank, "multiplier": multiplier, "initial_sha256": digest_initial_state(network),
            "groups": groups, "amp": bool(self.amp), "device": str(self.device),
            "train_inventory": image_inventory(self.train_loader.dataset.im_files),
            "val_inventory": image_inventory(self.test_loader.dataset.im_files),
        }
        (self.save_dir / f"optimizer_rank_{rank}.json").write_text(json.dumps(report, indent=2))
        self.track_optimizer_steps(base, network, rank)
        return base

    def track_optimizer_steps(self, optimizer, network, rank):
        # GradScaler calls these hooks only for successful, unscaled optimizer steps.
        layers = (6, 8, 11, 12, 13, 14, 15, 16, 17, 18, 19)
        watched = {}
        for index in layers:
            choices = (
                (name, param) for name, param in network.model[index].named_parameters()
                if param.requires_grad and param.ndim >= 2
            )
            name, parameter = next(choices)
            watched[f"model.{index}.{name}"] = parameter

        path = self.save_dir / f"optimizer_steps_rank_{rank}.jsonl"
        epoch_seen = set()
        sample = None
        steps = 0
        start_steps = 0
        start_attempts = 0
        start_scale = None

        def start_epoch(_trainer):
            nonlocal start_steps, start_attempts, start_scale
            self.batch_order = hashlib.sha256()
            start_steps = steps
            start_attempts = self.optimizer_attempts
            start_scale = self.scaler.get_scale()

        def finish_epoch(_trainer):
            with (self.save_dir / f"training_epochs_rank_{rank}.jsonl").open("a", encoding="utf-8") as stream:
                stream.write(json.dumps({
                    "epoch": int(self.epoch) + 1,
                    "attempted_steps": self.optimizer_attempts - start_attempts,
                    "successful_steps": steps - start_steps,
                    "amp_scale_start": start_scale,
                    "amp_scale_end": self.scaler.get_scale(),
                    "batch_order_sha256": self.batch_order.hexdigest(),
                }) + "\n")

        def before(opt, _args, _kwargs):
            nonlocal sample, steps
            steps += 1
            epoch = int(self.epoch)
            if epoch in epoch_seen and steps > 3:
                return
            epoch_seen.add(epoch)
            batch_values = {
                key: (value.detach().clone(), value.grad.detach().float().square().mean().sqrt())
                for key, value in watched.items() if value.grad is not None
            }
            sample = {
                "step": steps, "epoch": epoch + 1, "lr": [g["lr"] for g in opt.param_groups],
                "parameters": batch_values,
            }

        def after(_opt, _args, _kwargs):
            nonlocal sample
            if sample is None:
                return
            measurements = {}
            for key, (before_value, grad_rms) in sample.pop("parameters").items():
                difference = (watched[key].detach() - before_value).float()
                change = difference.square().mean().sqrt().item()
                scale = before_value.float().square().mean().sqrt().item()
                measurements[key] = {"grad_rms": float(grad_rms),
                                     "update_rms": change, "update_over_weight": change / max(scale, 1e-12)}
                if not all(math.isfinite(value) for value in measurements[key].values()):
                    raise RuntimeError(f"Nonfinite update or gradient: {key}")
            with path.open("a", encoding="utf-8") as stream:
                stream.write(json.dumps({**sample, "parameters": measurements}, allow_nan=False) + "\n")
            sample = None

        optimizer.register_step_pre_hook(before)
        optimizer.register_step_post_hook(after)
        self.add_callback("on_train_epoch_start", start_epoch)
        self.add_callback("on_train_epoch_end", finish_epoch)


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/decision.py
"""Frozen decision rules and evidence handling for the YOLO11 experiment."""

import hashlib
import json
import math
from pathlib import Path
from statistics import mean


SEEDS = (3407, 3408, 3409, 3410, 3411)
OPERATING_SEED = SEEDS[0]
ARMS = {"control": 1.0, "treatment": 1.5}
FLOOR = (306, 367)
MIN_RECALL_GAIN = 0.02
MAX_AP_DROP = 0.001
DATA_SHA = "026a2afb617254b9dd2b9461651bce6f74295d18f29d52a790bf755d32944f65"
B1_SHA = "52f4490280ed92e378d2d9936256cb286a1b2fd102114c0f226324b8935553d7"
C0_HASHES = {
    3407: "472e60d2cc68741abdeebfe25a1e50742eaef68418f3013930224395f54012ff",
    3408: "bdd03836ab96134336ff531d66b8207a8456d8c4cb0f222612c44727a2fbd22a",
    3409: "1a64133a3ab17326073e752a0df98ab0d5e878480c98f34b9af1ed620913d641",
    3410: "214a6459451bf58c351c6e71e38a5bfc71318bbe4afcbc1661c5f7ca69a4d3af",
    3411: "846d7c0f0fc2ab8b447c39eeb1a40b8c7877f5a4080d72738539050bb1dd101c",
}


def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path, value):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def write_receipt(root, filename, **metadata):
    files = {
        path.relative_to(root).as_posix(): sha256(path)
        for path in sorted(root.rglob("*")) if path.is_file() and path.name != filename
    }
    write_json(root / filename, {**metadata, "files": files})


def verify_receipt(path, required=(), verify_all=False):
    receipt = read_json(path)
    files = receipt["files"]
    for name in set(files if verify_all else required):
        candidate = (path.parent / name).resolve()
        if (not candidate.is_relative_to(path.parent.resolve()) or name not in files
                or not candidate.is_file() or sha256(candidate) != files[name]):
            raise ValueError(f"Missing or modified evidence in {path.name}: {name}")
    return receipt


def counts(tp, fp, support):
    if min(tp, fp) < 0 or not 0 < support or tp > support:
        raise ValueError("Invalid detection counts.")
    return {"tp": tp, "fp": fp, "fn": support - tp,
            "precision": tp / (tp + fp) if tp + fp else None, "recall": tp / support}


def meets_floor(point):
    return (point is not None and point["tp"] > 0
            and point["tp"] * FLOOR[1] >= (point["tp"] + point["fp"]) * FLOOR[0])


def minimum(value, limit):
    return value >= limit or math.isclose(value, limit, rel_tol=0, abs_tol=1e-12)


def compare(candidate, reference):
    point, baseline = candidate["operating_point"], reference["operating_point"]
    return {
        "precision_pass": meets_floor(point),
        "delta_recall": None if point is None or baseline is None else point["recall"] - baseline["recall"],
        "delta_map50_95": candidate["native_metrics"]["mAP50_95"] - reference["native_metrics"]["mAP50_95"],
    }


def seed_gate(candidates, references):
    rows = {seed: compare(candidates[seed], references[seed]) for seed in SEEDS}
    recall = [row["delta_recall"] for row in rows.values() if row["delta_recall"] is not None]
    mean_recall = mean(recall) if len(recall) == len(SEEDS) else None
    mean_ap = mean(row["delta_map50_95"] for row in rows.values())
    positive = sum(change > 0 for change in recall)
    return {
        "by_seed": rows, "mean_delta_recall": mean_recall, "mean_delta_map50_95": mean_ap,
        "positive_seeds": positive,
        "pass": mean_recall is not None and minimum(mean_recall, MIN_RECALL_GAIN)
        and positive >= 4 and all(row["precision_pass"] for row in rows.values())
        and minimum(mean_ap, -MAX_AP_DROP),
    }


def lock_validation(original, stock, control, treatment, validation_receipts):
    baseline = dict.fromkeys(SEEDS, original)
    if not meets_floor(original["operating_point"]):
        raise ValueError("The validated B1 reference has no eligible operating point.")
    candidates = {"C0": stock, "B1_loss": treatment}
    against_b1 = {name: seed_gate(rows, baseline) for name, rows in candidates.items()}
    against_control = seed_gate(treatment, control)
    eligible = []
    for name, rows in candidates.items():
        target = compare(rows[OPERATING_SEED], original)
        if (against_b1[name]["pass"] and (name == "C0" or against_control["pass"])
                and target["precision_pass"] and target["delta_recall"] > 0
                and minimum(target["delta_map50_95"], -MAX_AP_DROP)):
            eligible.append(name)
    ranking = sorted(eligible, key=lambda name: (
        -against_b1[name]["mean_delta_recall"],
        -against_b1[name]["mean_delta_map50_95"], name != "C0",
    ))
    test_models = {"B1": original, "C0": stock[OPERATING_SEED],
                   "B1_control": control[OPERATING_SEED], "B1_loss": treatment[OPERATING_SEED]}
    return {
        "dataset_sha256": DATA_SHA, "test_seed": OPERATING_SEED, "validation_ranking": ranking,
        "seed_comparisons_with_B1": against_b1, "loss_vs_control": against_control,
        "validation_receipts_sha256": validation_receipts,
        "test_models": {name: {"checkpoint_sha256": row["checkpoint_sha256"],
                               "threshold": (row["operating_point"] or {"threshold": None})["threshold"]}
                        for name, row in test_models.items()},
    }


def conclude(validation, rows):
    for name, locked in validation["test_models"].items():
        point = rows[name]["operating_point"]
        if (rows[name]["checkpoint_sha256"] != locked["checkpoint_sha256"]
                or (None if point is None else point["threshold"]) != locked["threshold"]):
            raise ValueError(f"Test used a different checkpoint or threshold for {name}.")
    comparisons = {name: compare(rows[name], rows["B1"]) for name in rows if name != "B1"}
    ranking = validation["validation_ranking"]
    winner = ranking[0] if ranking else None
    delta = comparisons.get(winner)
    confirmed = bool(delta and delta["precision_pass"] and delta["delta_recall"] is not None
                     and minimum(delta["delta_recall"], MIN_RECALL_GAIN)
                     and minimum(delta["delta_map50_95"], -MAX_AP_DROP))
    return {"selected": winner if confirmed else "B1", "validation_winner": winner,
            "confirmed_on_test": confirmed, "test_comparisons_with_B1": comparisons}


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/evaluate.py
"""Evaluate stock and DCRCF checkpoints with the same pinned replay and validator."""

from collections import Counter
from pathlib import Path

import math
import torch
import yaml
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionValidator
from ultralytics.utils.nms import non_max_suppression
from ultralytics.utils.ops import xywh2xyxy

from b1_lr_evaluate import THRESHOLDS, case_details, image_and_targets, match_targets, operating_point, write_csv
from decision import counts, sha256, write_json
from gdut_data import NAMES
from warm_metrics import export_metrics
from warm_protocol import EVALUATION


def build_curve(detections, support):
    curve = []
    tp = fp = 0
    for confidence, correct in sorted(detections, key=lambda row: -row[0]):
        tp += int(correct)
        fp += int(not correct)
        point = {"threshold": confidence, **counts(tp, fp, support)}
        if curve and curve[-1]["threshold"] == confidence:
            curve[-1] = point
        else:
            curve.append(point)
    return curve


def native_metrics(checkpoint, prepared, records, split, output, device):
    seen = []

    class AuditedValidator(DetectionValidator):
        def update_metrics(self, predictions, batch):
            seen.extend(Path(path).stem for path in batch["im_file"])
            super().update_metrics(predictions, batch)

    data = output / "evaluation.yaml"
    source = str(prepared / "base/images" / split)
    # Ultralytics requires train and val keys even when only the test split is read.
    data.write_text(yaml.safe_dump({"train": source, "val": source, split: source, "names": NAMES}))
    result = YOLO(str(checkpoint)).val(validator=AuditedValidator, data=str(data), split=split,
                                      device=device, project=str(output), name="native", save_json=True, **EVALUATION)
    support = Counter(int(box[0]) for record in records for box in record["boxes"])
    if sorted(seen) != sorted(row["id"] for row in records):
        raise ValueError("Native validator read a different set of images.")
    if result.nt_per_class.tolist() != [support[index] for index in range(len(NAMES))]:
        raise ValueError("Native validator read a different annotation inventory.")
    summary, classes, definitions = export_metrics(result)
    if not all(math.isfinite(summary[key]) for key in ("precision", "recall", "mAP50", "mAP50_95")):
        raise ValueError("The native evaluator computed nonfinite metrics.")
    for row in classes:
        row["support"] = support[row["class_id"]]
    if not classes:
        raise ValueError("The native evaluator returned no class metrics.")
    write_csv(output / "per_class.csv", classes)
    return summary, classes, definitions


@torch.inference_mode()
def replay(checkpoint, prepared, records, split, output, device, threshold):
    network = YOLO(str(checkpoint)).model.to(device).float().eval()
    if [network.names[index] for index in range(len(NAMES))] != NAMES:
        raise ValueError("Checkpoint names do not match the frozen GDUT classes.")
    if network.model[-1].stride.tolist() != [8, 16, 32]:
        raise ValueError("Unexpected Detect pyramid for the shared replay.")
    support = sum(box[0] == 4 for record in records for box in record["boxes"])
    image_dir = prepared / "base/images" / split
    ranked, images, errors = [], [], []
    for index, record in enumerate(records, 1):
        image, targets = image_and_targets(record, image_dir, device)
        prediction = network(image)
        raw = prediction[0] if isinstance(prediction, (tuple, list)) else prediction
        if list(raw.shape) != [1, 9, 8400] or not torch.isfinite(raw).all():
            raise ValueError(f"Invalid decoded tensor for {record['id']}.")
        detections = non_max_suppression(raw.clone(), conf_thres=0.001, iou_thres=0.70, nc=5,
                                         max_det=300, multi_label=False)[0].float().cpu()
        ordered, pairs = match_targets(detections, targets)
        by_detection = dict(pairs)
        rows = [{"confidence": float(detections[i, 4]), "target_index": by_detection.get(i)} for i in ordered]
        ranked.extend((row["confidence"], row["target_index"] is not None) for row in rows)
        images.append({"image_id": record["id"], "support": len(targets), "detections": rows})
        errors.extend(case_details(record["id"], targets, xywh2xyxy(raw[0, :4].T).float().cpu(),
                                   raw[0, 4:].float().cpu(), detections[detections[:, 4] >= 0.25]))
        if index % 128 == 0:
            print(f"{split} replay {index}/{len(records)}", flush=True)
    if len(errors) != support:
        raise ValueError("Incomplete class none error inventory.")

    curve = build_curve(ranked, support)

    def at(value):
        point = next((row for row in reversed(curve) if row["threshold"] >= value), None)
        return {**(point or counts(0, 0, support)), "threshold": value}

    point = operating_point(curve) if split == "val" else (None if threshold is None else at(threshold))
    fixed = [at(value) for value in THRESHOLDS]
    fixed_025 = next(row for row in fixed if row["threshold"] == 0.25)
    if fixed_025["tp"] != sum(row["result"] == "matched" for row in errors):
        raise ValueError("Per-box outcomes and replay TP disagree.")
    write_csv(output / "none_pr.csv", fixed)
    write_csv(output / "none_pr_curve.csv", curve, ("threshold", "precision", "recall", "tp", "fp", "fn"))
    write_csv(output / "none_error_breakdown.csv", errors)
    write_json(output / "none_by_image.json", images)
    groups = Counter(("small" if row["sqrt_area_input_px"] < 32 else "other",
                      row["pyramid_level"] or "no_iou50", row["result"]) for row in errors)
    return {"fixed_025": fixed_025, "operating_point": point, "images": len(images), "support_none": support,
            "errors_by_size_pyramid": [{"size": size, "level": level, "reason": result, "count": count}
                                       for (size, level, result), count in sorted(groups.items())]}


def evaluate(checkpoint, prepared, records, split, output, device, threshold=None):
    output.mkdir(parents=True)
    diagnosis = replay(checkpoint, prepared, records, split, output,
                       "cpu" if device == "cpu" else f"cuda:{device}", threshold)
    native, classes, definitions = native_metrics(checkpoint, prepared, records, split, output, device)
    result = {**diagnosis, "split": split, "checkpoint_sha256": sha256(checkpoint),
              "native_metrics": native, "per_class": classes, "definitions": definitions}
    write_json(output / "summary.json", result)
    return result


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/loss_trainer.py
"""Train the paired loss experiment on the verified B1 trainer and graph."""

import os

import torch
from ultralytics.utils.loss import v8DetectionLoss
from ultralytics.utils.torch_utils import unwrap_model

from b1_lr_trainer import PairedLRTrainer
from baseline_model import PaperBaselineModel
from decision import ARMS, write_json
from positive_loss import positive_bce


class PositiveNoneModel(PaperBaselineModel):
    def init_criterion(self):
        criterion = super().init_criterion()
        if (type(criterion) is not v8DetectionLoss or criterion.nc != 5
                or self.names[4] != "none" or self.baseline_id != "yolo11_dcrcf"):
            raise ValueError("Unexpected GDUT detection loss or class order.")
        if self.none_positive_weight != 1.0:
            criterion.bce = positive_bce(self.none_positive_weight, criterion.device)
        return criterion


class PositiveNoneTrainer(PairedLRTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        if type(model) is not PaperBaselineModel:
            raise ValueError("The archived trainer returned a different model type.")
        model.__class__ = PositiveNoneModel
        model.none_positive_weight = ARMS[os.environ["B1_NONE_ARM"]]
        return model

    def _setup_train(self):
        super()._setup_train()
        rank = int(os.environ["RANK"])

        def record_effective_loss(trainer):
            path = trainer.save_dir / f"loss_rank_{rank}.json"
            if path.exists():
                return
            model = unwrap_model(trainer.model)
            bce = model.criterion.bce
            expected = [1.0] * 4 + [model.none_positive_weight]
            observed = [1.0] * 5 if bce.pos_weight is None else bce.pos_weight.detach().cpu().tolist()
            if observed != expected or bce.weight is not None or bce.reduction != "none":
                raise ValueError(f"Unexpected effective classification loss on rank {rank}.")
            if not all(torch.isfinite(value).all() for value in trainer.loss_items.values()):
                raise ValueError(f"Nonfinite AMP loss on rank {rank}.")
            write_json(path, {"rank": rank, "positive_weights": observed})

        self.add_callback("on_train_epoch_end", record_effective_loss)


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/positive_loss.py
"""Positive-only GDUT classification weight for the pinned BCE criterion."""

import torch


def positive_bce(weight, device):
    if weight == 1.0:
        return torch.nn.BCEWithLogitsLoss(reduction="none")
    positive_weights = torch.ones(5, device=device)
    positive_weights[4] = weight
    return torch.nn.BCEWithLogitsLoss(pos_weight=positive_weights, reduction="none")


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/workflow.py
"""Run one verified phase of the GDUT YOLO11 decision workflow."""

import argparse
from collections import Counter
import hashlib
import os
from pathlib import Path
import random
import shutil
import sys
from tempfile import mkdtemp

from decision import (ARMS, B1_SHA, C0_HASHES, DATA_SHA, OPERATING_SEED, SEEDS, conclude,
                      lock_validation, read_json, sha256, verify_receipt, write_json, write_receipt)


HERE = Path(__file__).resolve().parent
EXPERIMENT = "gdut-yolo11-positive-none-v1"
STAGE1_SHA = "ea174f78b6b768a2f436c4248a7b95e3001ca3ff47fe2d850a78c22f3e98a88c"
MIRROR_MANIFEST_SHA = "ce2a883e0b5050edb96d494218e85a9d16ea04b9a19dc5723c572cb502edfe9c"
HARDAUG_SHA = "7f6d2def5d356b03e0544df71e7a0fefe3fa0ada073fa25d70dfaceb101a37d6"
FIXED_A = (306, 61, 98)
NAMES = ["blue", "white", "yellow", "red", "none"]
RECIPE_FILES = ("decision.py", "evaluate.py", "loss_trainer.py", "positive_loss.py", "workflow.py",
                "b1_lr_evaluate.py", "b1_lr_run.py", "b1_lr_trainer.py")


def unique(paths, label):
    paths = sorted(set(paths))
    if len(paths) != 1:
        raise ValueError(f"{label}: expected one verified source, found {len(paths)}.")
    return paths[0]


def roots(args):
    return [path.resolve(strict=True) for path in args.input_root]


def locate(dataset_roots, filename):
    return sorted({path for root in dataset_roots for path in root.rglob(filename)})


def canonical_manifest(dataset_roots, work=None):
    candidates = {path: sha256(path) for path in locate(dataset_roots, "dataset_manifest.json")}
    original = [path for path, digest in candidates.items() if digest == DATA_SHA]
    if original:
        return unique(original, "GDUT source 360434104")
    mirror = unique((path for path, digest in candidates.items() if digest == MIRROR_MANIFEST_SHA),
                    "Verified equivalent GDUT preparation output")
    normalized = (mirror.read_bytes().replace(b"/gdut-hwd/GDUT-HWD/", b"/gdut-hwd/")
                  .replace(b'"notes": "GDUT test unchanged;',
                           b'"notes": "IITU group split unchanged. GDUT test unchanged;'))
    if hashlib.sha256(normalized).hexdigest() != DATA_SHA:
        raise ValueError("Source-path and note normalization did not reproduce the frozen manifest.")
    work = work or Path(mkdtemp(prefix="gdut-input-"))
    prepared = work / "prepared_gdut"
    prepared.mkdir()
    for item in ("base", "hardaug", "augmentation_manifest.json", "preparation_complete.json"):
        (prepared / item).symlink_to((mirror.parent / item).resolve(strict=True))
    manifest = prepared / "dataset_manifest.json"
    manifest.write_bytes(normalized)
    return manifest


def dataset(dataset_roots, split):
    from PIL import Image

    manifest = canonical_manifest(dataset_roots)
    registry = read_json(manifest)
    prepared = manifest.parent
    records = sorted((row for row in registry["records"] if row["split"] == split), key=lambda row: row["id"])
    ids = {row["id"] for row in records}
    image_dir, labels = prepared / "base/images" / split, prepared / "base/labels" / split
    expected = {"val": 316, "test": 1587}[split]
    if (registry["dataset"] != "gdut" or registry["names"] != NAMES
            or len(records) != expected or len(ids) != expected
            or {path.stem for path in image_dir.glob("*.jpg")} != ids
            or {path.stem for path in labels.glob("*.txt")} != ids):
        raise ValueError(f"Unexpected GDUT {split} inventory.")
    for row in records:
        uid = row["id"]
        with Image.open(image_dir / f"{uid}.jpg") as image:
            rgb = image.convert("RGB")
            digest = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
            if image.getexif().get(274, 1) != 1 or digest != row["pixel_hash"]:
                raise ValueError(f"Pixel or EXIF mismatch: {uid}.")
        actual = [line.split() for line in (labels / f"{uid}.txt").read_text().splitlines() if line.strip()]
        actual = Counter(tuple(round(float(value), 10) for value in box) for box in actual)
        expected_boxes = Counter(tuple(round(float(value), 10) for value in box) for box in row["boxes"])
        if actual != expected_boxes:
            raise ValueError(f"Label mismatch: {uid}.")
    if split == "val" and sum(box[0] == 4 for row in records for box in row["boxes"]) != 404:
        raise ValueError("GDUT validation must contain 404 class none boxes.")
    return prepared, records


def archived(directory_roots):
    path = unique((path for path in locate(directory_roots, "run_manifest.json")
                   if read_json(path).get("checkpoint_sha256") == B1_SHA), "Archived B1 checkpoint")
    manifest = read_json(path)
    weight = path.parent / "weights/best.pt"
    if (manifest["dataset_fingerprint"] != DATA_SHA or manifest["source_checkpoint_sha256"] != STAGE1_SHA
            or manifest["baseline_id"] != "yolo11_dcrcf" or sha256(weight) != B1_SHA):
        raise ValueError("The archived B1 weights or lineage differ from the reviewed source.")
    verify_archived_code(manifest)
    return weight


def verify_archived_code(manifest):
    source = manifest["code_sha256"]
    if len(source) != 17 or any(sha256(HERE / name) != digest for name, digest in source.items()):
        raise ValueError("Archived source module digest mismatch.")


def stock(directory_roots, seeds):
    receipt = unique((path for path in locate(directory_roots, "ADD_INPUT_READY.json")
                     if all(digest in path.read_text() for digest in C0_HASHES.values())), "C0 source receipt")
    paths = {}
    for seed in seeds:
        source = receipt.parent / f"C0-{seed}"
        weight = source / "weights/best.pt"
        manifest = read_json(source / "run_manifest.json")
        if (manifest["arm"] != "C0" or manifest["seed"] != seed
                or manifest["checkpoint_sha256"]["weights/best.pt"] != C0_HASHES[seed]
                or sha256(weight) != C0_HASHES[seed]):
            raise ValueError(f"The C0 weights or run metadata differ for seed {seed}.")
        paths[seed] = weight
    return paths


def recipe():
    return {name: sha256(HERE / name) for name in RECIPE_FILES}


def training_source(directory_roots):
    source = unique(directory_roots, "Kaggle input root")
    work = Path(mkdtemp(prefix="gdut-training-input-"))
    manifest = canonical_manifest(directory_roots, work)
    if not manifest.is_relative_to(work):
        return source
    # Stage the two small verified checkpoints so the archived lookup sees its
    # ordinary run layout. The prepared image and label mounts remain read-only.
    for digest, name in ((STAGE1_SHA, "stage1"), (B1_SHA, "B1")):
        run = unique((path for path in locate(directory_roots, "run_manifest.json")
                      if read_json(path).get("checkpoint_sha256") == digest), name)
        record = read_json(run)
        checkpoint = (run.parent / record["checkpoint"]).resolve()
        if not checkpoint.is_relative_to(run.parent.resolve()) or sha256(checkpoint) != digest:
            raise ValueError(f"Invalid checkpoint for {name}.")
        target = work / name / record["checkpoint"]
        target.parent.mkdir(parents=True)
        shutil.copyfile(run, work / name / "run_manifest.json")
        shutil.copyfile(checkpoint, target)
    return work


def evaluation_runtime(device):
    import torch
    import torchvision
    import ultralytics

    if ultralytics.__version__ != "8.4.115":
        raise RuntimeError("Use the pinned Ultralytics 8.4.115 runtime.")
    return {"torch": torch.__version__, "torchvision": torchvision.__version__, "cuda": torch.version.cuda,
            "ultralytics": ultralytics.__version__,
            "device": "cpu" if device == "cpu" else torch.cuda.get_device_name(int(device))}


def verify_a(result):
    fixed = result["fixed_025"]
    metadata = result["native_metrics"]
    if ((fixed["tp"], fixed["fp"], fixed["fn"]) != FIXED_A or result["support_none"] != 404
            or abs(metadata["mAP50_95"] - 0.5747310221861304) > 0.0005):
        raise ValueError(f"B1 did not reproduce A: {fixed}; mAP50-95={metadata['mAP50_95']}.")


def evaluate_base(args, root):
    from evaluate import evaluate

    directory_roots = roots(args)
    prepared, records = dataset(directory_roots, "val")
    runtime = evaluation_runtime(args.device)
    original = archived(directory_roots)
    weights = stock(directory_roots, SEEDS)
    baseline = evaluate(original, prepared, records, "val", root / "B1", args.device)
    verify_a(baseline)
    for seed, weight in weights.items():
        print(f"Evaluating C0 seed {seed}", flush=True)
        evaluate(weight, prepared, records, "val", root / f"C0-{seed}", args.device)
    write_receipt(root, "BASE_VALIDATED.json", experiment=EXPERIMENT, dataset_sha256=DATA_SHA,
                  runtime=runtime, evaluator_sha256=sha256(HERE / "evaluate.py"))


def child(args):
    import numpy as np
    import torch
    from loss_trainer import PositiveNoneTrainer
    from warm_protocol import training_settings
    from warm_runner import check_version, validate_saved_args

    check_version()
    if os.environ.get("B1_NONE_ARM") != args.arm or os.environ.get("B1_LR_MULTIPLIER") != "1.0":
        raise ValueError("Unexpected child configuration.")
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    options = training_settings("0,1", smoke=False)
    if args.phase == "smoke":
        options.update(epochs=2, workers=0, plots=False, close_mosaic=0)
    options.update(seed=args.seed, model=str(args.weight), data=str(args.data), project=str(args.project),
                   name=args.arm, exist_ok=False)
    PositiveNoneTrainer(overrides=options).train()
    validate_saved_args(args.project / args.arm / "args.yaml", options)


def train(args, root):
    from b1_lr_run import build_configs, lock_inputs, verify_arm, verify_pair
    from b1_lr_trainer import AMP_INITIAL_SCALE
    from warm_environment import runtime_snapshot
    from warm_launch import run_logged

    phase = "smoke" if args.mode == "smoke" else "full"
    if phase == "smoke" and args.seed != OPERATING_SEED:
        raise ValueError("Smoke uses the fixed validation seed 3407.")
    runtime = runtime_snapshot("0,1", "Tesla T4")
    source = training_source(roots(args))
    prepared, weight, _, historical, records, hardaug = lock_inputs(source)
    if hardaug != HARDAUG_SHA:
        raise ValueError("Mounted hardaug differs from the content digest of the verified B1 run.")
    if phase == "full":
        smoke_path = args.smoke_receipt or unique(
            (p for p in locate(roots(args), "SMOKE_READY.json") if read_json(p).get("experiment") == EXPERIMENT),
            "Verified loss smoke receipt")
        smoke = verify_receipt(smoke_path, ("launch.json", "control/summary.json", "treatment/summary.json"))
        if (smoke["experiment"] != EXPERIMENT or smoke["recipe"] != recipe()
                or smoke["hardaug_sha256"] != hardaug or smoke["runtime"] != runtime):
            raise ValueError("Smoke and full differ in code, runtime or hardaug: repeat smoke on the locked configuration.")
    configs, inventory = build_configs(prepared, records, root)
    write_json(root / "launch.json", {"experiment": EXPERIMENT, "phase": phase, "seed": args.seed,
               "runtime": runtime, "recipe": recipe(), "dataset_sha256": DATA_SHA,
               "hardaug_sha256": hardaug, "stage1_sha256": STAGE1_SHA, "baseline_sha256": B1_SHA,
               "amp_scale": AMP_INITIAL_SCALE, "positive_weights": ARMS})

    project = root / "training"
    evidence = {}
    for arm, factor in ARMS.items():
        command = [sys.executable, str(HERE / "workflow.py"), "child", "--weight", str(weight),
                   "--data", str(configs[phase]), "--project", str(project), "--arm", arm,
                   "--phase", phase, "--seed", str(args.seed)]
        os.environ.update(B1_NONE_ARM=arm, B1_LR_MULTIPLIER="1.0", PYTHONHASHSEED=str(args.seed))
        try:
            run_logged(command, root / f"train-{arm}.log")
        finally:
            os.environ.pop("B1_NONE_ARM", None)
            os.environ.pop("B1_LR_MULTIPLIER", None)
        evidence[arm] = verify_arm(project / arm, phase, 1.0, historical, inventory)
        for rank in (0, 1):
            observed = read_json(project / arm / f"loss_rank_{rank}.json")
            if observed != {"rank": rank, "positive_weights": [1.0] * 4 + [factor]}:
                raise ValueError(f"Loss evidence is incorrect for {arm} on rank {rank}.")
    verify_pair(evidence["control"], evidence["treatment"])
    for left, right in zip(evidence["control"][2], evidence["treatment"][2], strict=True):
        if left["groups"] != right["groups"]:
            raise ValueError("The two arms changed optimizer settings or groups.")
    checkpoints = {arm: sha256(row[0]) for arm, row in evidence.items()}
    meta = dict(experiment=EXPERIMENT, seed=args.seed, runtime=runtime, recipe=recipe(),
                dataset_sha256=DATA_SHA, hardaug_sha256=hardaug, checkpoint_sha256=checkpoints)
    write_receipt(root, "FULL_TRAINED.json" if phase == "full" else "SMOKE_TRAINED.json", **meta)
    score_trained(root, root, prepared, records["val"], args.device, phase, meta)


def score_trained(trained, root, prepared, records, device, phase, meta):
    from evaluate import evaluate
    from loss_trainer import PositiveNoneModel  # noqa: F401  (checkpoint class is serialized)

    for arm in ARMS:
        weight = trained / "training" / arm / "weights/best.pt"
        if sha256(weight) != meta["checkpoint_sha256"][arm]:
            raise ValueError(f"Training receipt does not match the {arm} checkpoint.")
        evaluate(weight, prepared, records, "val", root / arm, device)
    write_receipt(root, "PAIR_VALIDATED.json" if phase == "full" else "SMOKE_READY.json",
                  **meta, evaluator_runtime=evaluation_runtime(device))


def recover(args, root):
    path = args.trained_receipt.resolve(strict=True)
    receipt = verify_receipt(path, ("launch.json", *(f"training/{arm}/weights/best.pt" for arm in ARMS)))
    if receipt["experiment"] != EXPERIMENT or receipt["recipe"] != recipe() or receipt["dataset_sha256"] != DATA_SHA:
        raise ValueError("The completed training receipt is from another experiment.")
    prepared, records = dataset(roots(args), "val")
    score_trained(path.parent, root, prepared, records, args.device, "full",
                  {key: value for key, value in receipt.items() if key != "files"})


def make_lock(args, root):
    paths = roots(args)
    base_path = unique(locate(paths, "BASE_VALIDATED.json"), "Validated source comparison")
    summaries = ("B1/summary.json", *(f"C0-{seed}/summary.json" for seed in SEEDS))
    base = verify_receipt(base_path, summaries)
    if (base["experiment"] != EXPERIMENT or base["dataset_sha256"] != DATA_SHA
            or base["evaluator_sha256"] != sha256(HERE / "evaluate.py")):
        raise ValueError("Base validation receipt does not match the evaluator or dataset.")
    original = read_json(base_path.parent / summaries[0])
    verify_a(original)
    if original["checkpoint_sha256"] != B1_SHA:
        raise ValueError("Base receipt did not evaluate Thanh's archive.")
    c0 = {seed: read_json(base_path.parent / f"C0-{seed}/summary.json") for seed in SEEDS}
    if any(c0[seed]["checkpoint_sha256"] != C0_HASHES[seed] for seed in SEEDS):
        raise ValueError("A C0 validation summary did not use its frozen checkpoint.")
    controls, treatments, receipts = {}, {}, {"base": sha256(base_path)}
    training_runtime = None
    for path in locate(paths, "PAIR_VALIDATED.json"):
        pair = verify_receipt(path, ("control/summary.json", "treatment/summary.json"))
        seed = pair["seed"]
        if (pair["experiment"] != EXPERIMENT or pair["recipe"] != recipe()
                or pair["dataset_sha256"] != DATA_SHA or seed not in SEEDS or seed in controls):
            raise ValueError(f"Unexpected or duplicate paired receipt at {path}.")
        training_runtime = training_runtime or pair["runtime"]
        if pair["runtime"] != training_runtime or pair["evaluator_runtime"] != base["runtime"]:
            raise ValueError("The comparison requires a common evaluator runtime and like-for-like paired training.")
        rows = {arm: read_json(path.parent / arm / "summary.json") for arm in ARMS}
        if any(row["checkpoint_sha256"] != pair["checkpoint_sha256"][arm] for arm, row in rows.items()):
            raise ValueError("Pair summaries and checkpoints disagree.")
        controls[seed], treatments[seed] = rows["control"], rows["treatment"]
        receipts[str(seed)] = sha256(path)
    if sorted(controls) != list(SEEDS):
        raise ValueError("Five distinct complete paired validation receipts are required.")
    lock = lock_validation(original, c0, controls, treatments, receipts)
    lock.update(experiment=EXPERIMENT, evaluator_runtime=base["runtime"], recipe=recipe())
    write_json(root / "VALIDATION_LOCK.json", lock)


def final_evaluation(args, root):
    from evaluate import evaluate
    from loss_trainer import PositiveNoneModel  # noqa: F401  (checkpoint class is serialized)

    path = args.validation_lock.resolve(strict=True)
    if sha256(path) != args.lock_sha256:
        raise ValueError("Validation lock SHA-256 differs from the frozen decision input.")
    lock = read_json(path)
    if (lock["experiment"] != EXPERIMENT or lock["recipe"] != recipe()
            or lock["dataset_sha256"] != DATA_SHA or lock["evaluator_runtime"] != evaluation_runtime(args.device)):
        raise ValueError("Final test must use the frozen dataset, evaluator and runtime.")
    paths = roots(args)
    prepared, records = dataset(paths, "test")
    weights = {"B1": archived(paths), "C0": stock(paths, (OPERATING_SEED,))[OPERATING_SEED]}
    for arm, name in (("control", "B1_control"), ("treatment", "B1_loss")):
        digest = lock["test_models"][name]["checkpoint_sha256"]
        weights[name] = unique((p for p in locate(paths, "best.pt") if p.parent.parent.name == arm
                                and sha256(p) == digest), f"Frozen seed {OPERATING_SEED} {arm}")
    rows = {}
    for name, model in lock["test_models"].items():
        if sha256(weights[name]) != model["checkpoint_sha256"]:
            raise ValueError(f"Frozen checkpoint hash differs for {name}.")
        rows[name] = evaluate(weights[name], prepared, records, "test", root / name, args.device, model["threshold"])
    result = conclude(lock, rows)
    write_json(root / "decision.json", {**result, "validation": lock, "test": rows})
    write_receipt(root, "FINAL_EVALUATED.json", experiment=EXPERIMENT, dataset_sha256=DATA_SHA,
                  validation_lock_sha256=args.lock_sha256, selected=result["selected"])


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("mode", choices=("base", "smoke", "pair", "recover", "lock", "test", "child"))
    parser.add_argument("--input-root", type=Path, action="append")
    parser.add_argument("--output-base", type=Path, default=Path("/kaggle/working/yolo11_decision"))
    parser.add_argument("--device", choices=("0", "cpu"), default="0")
    parser.add_argument("--seed", type=int, choices=SEEDS, default=OPERATING_SEED)
    parser.add_argument("--smoke-receipt", type=Path)
    parser.add_argument("--trained-receipt", type=Path)
    parser.add_argument("--validation-lock", type=Path)
    parser.add_argument("--lock-sha256")
    parser.add_argument("--weight", type=Path)
    parser.add_argument("--data", type=Path)
    parser.add_argument("--project", type=Path)
    parser.add_argument("--arm", choices=ARMS)
    parser.add_argument("--phase", choices=("smoke", "full"))
    args = parser.parse_args()
    args.input_root = args.input_root or [Path("/kaggle/input")]
    if args.mode == "child":
        if not all((args.weight, args.data, args.project, args.arm, args.phase)):
            parser.error("Child mode requires --weight, --data, --project, --arm and --phase.")
        child(args)
        return
    if args.mode == "recover" and not args.trained_receipt:
        parser.error("Recovery requires --trained-receipt.")
    if args.mode == "test" and not (args.validation_lock and args.lock_sha256):
        parser.error("Test requires --validation-lock and the independently frozen --lock-sha256.")
    args.output_base = args.output_base.resolve()
    args.output_base.mkdir(parents=True, exist_ok=True)
    root = Path(mkdtemp(prefix=f"{args.mode}-{args.seed}-", dir=args.output_base))
    print(f"Evidence directory: {root}", flush=True)
    action = {"base": evaluate_base, "smoke": train, "pair": train, "recover": recover,
              "lock": make_lock, "test": final_evaluation}
    action[args.mode](args, root)
    print(f"Verified phase {args.mode} at {root}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
smoke_dir = Path('/kaggle/working/verified_smoke')
for name, content in {'SMOKE_READY.json': b'{\n  "experiment": "gdut-yolo11-positive-none-v1",\n  "seed": 3407,\n  "runtime": {\n    "hardware": {\n      "gpus": [\n        {\n          "name": "Tesla T4",\n          "memory_bytes": 15636037632,\n          "capability": [\n            7,\n            5\n          ]\n        },\n        {\n          "name": "Tesla T4",\n          "memory_bytes": 15636037632,\n          "capability": [\n            7,\n            5\n          ]\n        }\n      ],\n      "cpu_architecture": "x86_64"\n    },\n    "software": {\n      "packages": {\n        "torch": "2.11.0+cu130",\n        "torchvision": "0.26.0+cu130",\n        "ultralytics": "8.4.115",\n        "albumentations": "2.0.8",\n        "numpy": "2.1.3",\n        "scipy": "1.16.3",\n        "cv2": "4.14.0",\n        "PIL": "13.0.0"\n      },\n      "python": "3.13.16",\n      "cuda": "13.0",\n      "cudnn": 92700,\n      "system": "Linux",\n      "architecture": "x86_64"\n    },\n    "host": {\n      "cpu": "x86_64",\n      "cpu_count": 4\n    }\n  },\n  "recipe": {\n    "decision.py": "074f4c40b9eb67c26d0174f56b1a7c7d0d78816d45de93e01a43a96a82fd4848",\n    "evaluate.py": "6a2e566176b3ea20c2a48ffd1efa0c1bde84c945c1b76497415a43dc841b0479",\n    "loss_trainer.py": "4cac2ebdfa2190d0b81c02c8fa4cf20d6815dd3b6aa1965dc13b4573fbe7e02e",\n    "positive_loss.py": "79ff9219d4369afa6f5ea7ea3a17bbe808ef329e68dd90911a934dc632b8634c",\n    "workflow.py": "1761f83c3deb018041aa25b000a4e06b89f187a89804da80ecd129357b67a4ea",\n    "b1_lr_evaluate.py": "2fa741325c87c08c3643800ad73385cfe7d3c5a5a89b53f7e45dd24790bf442f",\n    "b1_lr_run.py": "6b3848263fd06ee566b6f4a6efe411e973289b06fe5c67cad8d38596ba90de4b",\n    "b1_lr_trainer.py": "86de2fbde673afcb6047a7b4445860f417db9faa903aa54c1a9fab71cb3d9843"\n  },\n  "dataset_sha256": "026a2afb617254b9dd2b9461651bce6f74295d18f29d52a790bf755d32944f65",\n  "hardaug_sha256": "7f6d2def5d356b03e0544df71e7a0fefe3fa0ada073fa25d70dfaceb101a37d6",\n  "checkpoint_sha256": {\n    "control": "a9dc4b05f3bdf6d1beaf20e9eef855c56438ef4dc6a4c0b08eb2b32628d4224e",\n    "treatment": "f8cd51a942524110c111765e58a7b639f5c06c899dd8e32cb2856be4b9c47eb0"\n  },\n  "evaluator_runtime": {\n    "torch": "2.11.0+cu130",\n    "torchvision": "0.26.0+cu130",\n    "cuda": "13.0",\n    "ultralytics": "8.4.115",\n    "device": "Tesla T4"\n  },\n  "files": {\n    "SMOKE_TRAINED.json": "97676aa759f4935641252997689741ebfca36f9eeaced1fa66695e9a9e447c86",\n    "control/evaluation.yaml": "5c56fedb6437c80f2b77df0265abde59755679c6c88a22fa4e5467ea468fe6c6",\n    "control/native/BoxF1_curve.png": "30c0123c7d7c41e109649556de0273f5a95be62038c3f3d8fe6b4d54c8c0e06c",\n    "control/native/BoxPR_curve.png": "956a30c793aea9fdaf1b90c2725454ad279099ce4842427c9c4b0e9a39edb818",\n    "control/native/BoxP_curve.png": "1c5611cb98da9a9234e274f8ac019c531d9623903f542d792071614c4598c539",\n    "control/native/BoxR_curve.png": "7197a7518a419edccfbec534784b47091677a847434cca315641b48d3fcdda96",\n    "control/native/confusion_matrix.png": "eed37de479fa6a2ed0f6491dc2ee17a19aae640af3bfb5f5003d1cb2b3fef859",\n    "control/native/confusion_matrix_normalized.png": "c2a7be442429adeaf38709db6ae42909a7f1c673ee048291f4812a29c1865798",\n    "control/native/val_batch0_labels.jpg": "ebe51a2adaf3e8d0071d268d79b783e4c61470e2d6fc5f5277b2ed39ec791bfc",\n    "control/native/val_batch0_pred.jpg": "5b4281c1b31f2898fbda471300ba18c92532f09da83c167bc9f20ad59c935b4d",\n    "control/native/val_batch1_labels.jpg": "f3b5cb5472032e606bae0ebf0d365d3d68a7c90d8a0edfc0683aafe5ac167c64",\n    "control/native/val_batch1_pred.jpg": "52127e706ea48b71ec5592a874ca4e48594a8ba61399e83c3be1b6d37488c900",\n    "control/native/val_batch2_labels.jpg": "f4a1e9c32c62da4356e4dfbdbf77a9bf37f4ad91be1645d1227b2aa26e264cd6",\n    "control/native/val_batch2_pred.jpg": "8b74fbb139d22a501aefe0ddcba5f38be31373aa6a77d5e8989a0d109b8c1928",\n    "control/none_by_image.json": "04984d76189bd5133e17c330af9806d21ac6924c5130156a242b27eae449080d",\n    "control/none_error_breakdown.csv": "63b758829b2fd5023035dc2e0a884aa4a06f5d65c2b7954734a251411c47237c",\n    "control/none_pr.csv": "69ed7bf979289d58c0c5d79f2f636355b55e7b4b40b3e2b3cedfded9223b7d32",\n    "control/none_pr_curve.csv": "a7a81f991bdeb5f7cee4a8826f0a5c3c9832c8f557e98fbc5dc5153a6515c3d2",\n    "control/per_class.csv": "c40a723be81efc85d06a233c8b573841c7ee2d8152c23206c54879fa8b991420",\n    "control/summary.json": "63384623244f0c73d207407467f81ec5e1afd9e997763ad173dce2ff6d89901b",\n    "full.yaml": "d5cd4f8f8f54e5b68e45f243ad15874568d683aedd8eba270e7c62dec3e6bf1e",\n    "gdut_aug.yaml": "0fd3836a110bf16adae8e74136b9626c61a93c69fc587dfaa6449b482c2217a2",\n    "launch.json": "2442f3f2846b08fc595ded7fd0d001cb63a40c315adcca43395dc059aeeff53f",\n    "smoke.yaml": "65bf3fc61e216e4bce32245e400f89a4109112227e9b1599edb4b2e2a527d7cd",\n    "smoke_train.txt": "531fba44ea994801a96b6979a0c30ecfe2837083188ca475b2ece57a1a037ef3",\n    "train-control.log": "be8aad5b7239126e72c262c0700c07fe26dae8b5bb4d5abba72fb0d13152d189",\n    "train-treatment.log": "cf49dbc8c37fdd07f402c4c4a40c1383c1c142c61ac4a1e2eab3a166c8dc96a1",\n    "training/control/architecture_rank_0.json": "854cb89d181474e2b1a00e92146a092531dcaac87fe464c0497b9f080dd659e3",\n    "training/control/architecture_rank_1.json": "3fe536fbb8e6fa4672cdab897df22322e6cc037754a6bfc9db9144b9429b2945",\n    "training/control/args.yaml": "2de27bc7478d37bd825373db3646f8fbb82e01885e00688badf5e13a5fe7c58d",\n    "training/control/detect_input_rank_0.json": "b0c70aa26379d8c1ef10df4f43ba9c029f9c8940d7a3d95a5f702b8fe914be1e",\n    "training/control/detect_input_rank_1.json": "b0c70aa26379d8c1ef10df4f43ba9c029f9c8940d7a3d95a5f702b8fe914be1e",\n    "training/control/loss_rank_0.json": "cc6c290e4d4a1a64e56ec7be490f324631dc82525b50289fa1d742ba4c213450",\n    "training/control/loss_rank_1.json": "a9e5074e97d8d5019c578a962618cb8693436f880e6c54d78bc4159afecc6c39",\n    "training/control/optimizer_rank_0.json": "9c2e5aed1cf7beb1b4d8244b852b43ebc0062f008527d9eb1b542db7f84e232a",\n    "training/control/optimizer_rank_1.json": "16b90626107d9dbc505ac0645fd399e333dd2bd4d12aacfc53f384ef50467c21",\n    "training/control/optimizer_steps_rank_0.jsonl": "41f52f9b672f93cc32cb238a100f3908c8f4daf8dcba59dab4366e49d6dd9b6a",\n    "training/control/optimizer_steps_rank_1.jsonl": "41f52f9b672f93cc32cb238a100f3908c8f4daf8dcba59dab4366e49d6dd9b6a",\n    "training/control/results.csv": "479958a9d97046be59c3c1eb49f4952b9927755ec9dbb0d8f497ae4dbf22a2de",\n    "training/control/training_complete.json": "04065dfef3d13d6e2c332a58297a364f2ac3656bddbaa5f63c8ab1e4cd9c837c",\n    "training/control/training_epochs_rank_0.jsonl": "b1274788669faa4f022cd81473d21b19fb5142412caec4e040dbcf1d8060ae69",\n    "training/control/training_epochs_rank_1.jsonl": "26fa35cfee3a20e71e5ea200cdcb745b24d31b53fe54ec5320ead9b462bec4e1",\n    "training/control/weights/best.pt": "a9dc4b05f3bdf6d1beaf20e9eef855c56438ef4dc6a4c0b08eb2b32628d4224e",\n    "training/control/weights/last.pt": "84d23c2334dc62b6083a86c7ff9c27a30145e707c38a158e450f947f5591f681",\n    "training/treatment/architecture_rank_0.json": "854cb89d181474e2b1a00e92146a092531dcaac87fe464c0497b9f080dd659e3",\n    "training/treatment/architecture_rank_1.json": "3fe536fbb8e6fa4672cdab897df22322e6cc037754a6bfc9db9144b9429b2945",\n    "training/treatment/args.yaml": "22673ca1367020150fff399050d0087e8caca762e701b27a543d0ff78d242ade",\n    "training/treatment/detect_input_rank_0.json": "b0c70aa26379d8c1ef10df4f43ba9c029f9c8940d7a3d95a5f702b8fe914be1e",\n    "training/treatment/detect_input_rank_1.json": "b0c70aa26379d8c1ef10df4f43ba9c029f9c8940d7a3d95a5f702b8fe914be1e",\n    "training/treatment/loss_rank_0.json": "f02e4933714d5131b28ba617f3fe8ae2786b1b2e93320d490751c63fdc031db0",\n    "training/treatment/loss_rank_1.json": "2969adcf723342dec43f5292b7689304223d071a08e6a669e405ebeae5b10134",\n    "training/treatment/optimizer_rank_0.json": "9c2e5aed1cf7beb1b4d8244b852b43ebc0062f008527d9eb1b542db7f84e232a",\n    "training/treatment/optimizer_rank_1.json": "16b90626107d9dbc505ac0645fd399e333dd2bd4d12aacfc53f384ef50467c21",\n    "training/treatment/optimizer_steps_rank_0.jsonl": "06af9a10a763234d2861807c274c90dacf0deb74244b430c39efb4959998bce1",\n    "training/treatment/optimizer_steps_rank_1.jsonl": "06af9a10a763234d2861807c274c90dacf0deb74244b430c39efb4959998bce1",\n    "training/treatment/results.csv": "8bc900a367fdc6d9a1f1653ca8ed27ea5b8d6d471e8a88f96c180606a0ffe5d7",\n    "training/treatment/training_complete.json": "04065dfef3d13d6e2c332a58297a364f2ac3656bddbaa5f63c8ab1e4cd9c837c",\n    "training/treatment/training_epochs_rank_0.jsonl": "b1274788669faa4f022cd81473d21b19fb5142412caec4e040dbcf1d8060ae69",\n    "training/treatment/training_epochs_rank_1.jsonl": "26fa35cfee3a20e71e5ea200cdcb745b24d31b53fe54ec5320ead9b462bec4e1",\n    "training/treatment/weights/best.pt": "f8cd51a942524110c111765e58a7b639f5c06c899dd8e32cb2856be4b9c47eb0",\n    "training/treatment/weights/last.pt": "b6c494f1c6987dcfb5d0835f2517d0b9ba31c2d3367fbecf3d9a9d2e96ea2b68",\n    "treatment/evaluation.yaml": "5c56fedb6437c80f2b77df0265abde59755679c6c88a22fa4e5467ea468fe6c6",\n    "treatment/native/BoxF1_curve.png": "30c0123c7d7c41e109649556de0273f5a95be62038c3f3d8fe6b4d54c8c0e06c",\n    "treatment/native/BoxPR_curve.png": "956a30c793aea9fdaf1b90c2725454ad279099ce4842427c9c4b0e9a39edb818",\n    "treatment/native/BoxP_curve.png": "1c5611cb98da9a9234e274f8ac019c531d9623903f542d792071614c4598c539",\n    "treatment/native/BoxR_curve.png": "7197a7518a419edccfbec534784b47091677a847434cca315641b48d3fcdda96",\n    "treatment/native/confusion_matrix.png": "eed37de479fa6a2ed0f6491dc2ee17a19aae640af3bfb5f5003d1cb2b3fef859",\n    "treatment/native/confusion_matrix_normalized.png": "c2a7be442429adeaf38709db6ae42909a7f1c673ee048291f4812a29c1865798",\n    "treatment/native/val_batch0_labels.jpg": "ebe51a2adaf3e8d0071d268d79b783e4c61470e2d6fc5f5277b2ed39ec791bfc",\n    "treatment/native/val_batch0_pred.jpg": "5b4281c1b31f2898fbda471300ba18c92532f09da83c167bc9f20ad59c935b4d",\n    "treatment/native/val_batch1_labels.jpg": "f3b5cb5472032e606bae0ebf0d365d3d68a7c90d8a0edfc0683aafe5ac167c64",\n    "treatment/native/val_batch1_pred.jpg": "52127e706ea48b71ec5592a874ca4e48594a8ba61399e83c3be1b6d37488c900",\n    "treatment/native/val_batch2_labels.jpg": "f4a1e9c32c62da4356e4dfbdbf77a9bf37f4ad91be1645d1227b2aa26e264cd6",\n    "treatment/native/val_batch2_pred.jpg": "8b74fbb139d22a501aefe0ddcba5f38be31373aa6a77d5e8989a0d109b8c1928",\n    "treatment/none_by_image.json": "04984d76189bd5133e17c330af9806d21ac6924c5130156a242b27eae449080d",\n    "treatment/none_error_breakdown.csv": "8e2be62ffbec73db9474c8e4b07d9e1e6376ea6a2899935fb39a6f02085a6872",\n    "treatment/none_pr.csv": "69ed7bf979289d58c0c5d79f2f636355b55e7b4b40b3e2b3cedfded9223b7d32",\n    "treatment/none_pr_curve.csv": "a7a81f991bdeb5f7cee4a8826f0a5c3c9832c8f557e98fbc5dc5153a6515c3d2",\n    "treatment/per_class.csv": "c40a723be81efc85d06a233c8b573841c7ee2d8152c23206c54879fa8b991420",\n    "treatment/summary.json": "03c9baa62d4608b3404d6aa5d5cf4d1db408f7a50fdaa3398e779257ac9c7d18"\n  }\n}\n', 'launch.json': b'{\n  "experiment": "gdut-yolo11-positive-none-v1",\n  "phase": "smoke",\n  "seed": 3407,\n  "runtime": {\n    "hardware": {\n      "gpus": [\n        {\n          "name": "Tesla T4",\n          "memory_bytes": 15636037632,\n          "capability": [\n            7,\n            5\n          ]\n        },\n        {\n          "name": "Tesla T4",\n          "memory_bytes": 15636037632,\n          "capability": [\n            7,\n            5\n          ]\n        }\n      ],\n      "cpu_architecture": "x86_64"\n    },\n    "software": {\n      "packages": {\n        "torch": "2.11.0+cu130",\n        "torchvision": "0.26.0+cu130",\n        "ultralytics": "8.4.115",\n        "albumentations": "2.0.8",\n        "numpy": "2.1.3",\n        "scipy": "1.16.3",\n        "cv2": "4.14.0",\n        "PIL": "13.0.0"\n      },\n      "python": "3.13.16",\n      "cuda": "13.0",\n      "cudnn": 92700,\n      "system": "Linux",\n      "architecture": "x86_64"\n    },\n    "host": {\n      "cpu": "x86_64",\n      "cpu_count": 4\n    }\n  },\n  "recipe": {\n    "decision.py": "074f4c40b9eb67c26d0174f56b1a7c7d0d78816d45de93e01a43a96a82fd4848",\n    "evaluate.py": "6a2e566176b3ea20c2a48ffd1efa0c1bde84c945c1b76497415a43dc841b0479",\n    "loss_trainer.py": "4cac2ebdfa2190d0b81c02c8fa4cf20d6815dd3b6aa1965dc13b4573fbe7e02e",\n    "positive_loss.py": "79ff9219d4369afa6f5ea7ea3a17bbe808ef329e68dd90911a934dc632b8634c",\n    "workflow.py": "1761f83c3deb018041aa25b000a4e06b89f187a89804da80ecd129357b67a4ea",\n    "b1_lr_evaluate.py": "2fa741325c87c08c3643800ad73385cfe7d3c5a5a89b53f7e45dd24790bf442f",\n    "b1_lr_run.py": "6b3848263fd06ee566b6f4a6efe411e973289b06fe5c67cad8d38596ba90de4b",\n    "b1_lr_trainer.py": "86de2fbde673afcb6047a7b4445860f417db9faa903aa54c1a9fab71cb3d9843"\n  },\n  "dataset_sha256": "026a2afb617254b9dd2b9461651bce6f74295d18f29d52a790bf755d32944f65",\n  "hardaug_sha256": "7f6d2def5d356b03e0544df71e7a0fefe3fa0ada073fa25d70dfaceb101a37d6",\n  "stage1_sha256": "ea174f78b6b768a2f436c4248a7b95e3001ca3ff47fe2d850a78c22f3e98a88c",\n  "baseline_sha256": "52f4490280ed92e378d2d9936256cb286a1b2fd102114c0f226324b8935553d7",\n  "amp_scale": 16.0,\n  "positive_weights": {\n    "control": 1.0,\n    "treatment": 1.5\n  }\n}\n', 'control/summary.json': b'{\n  "fixed_025": {\n    "tp": 0,\n    "fp": 0,\n    "fn": 404,\n    "precision": null,\n    "recall": 0.0,\n    "threshold": 0.25\n  },\n  "operating_point": null,\n  "images": 316,\n  "support_none": 404,\n  "errors_by_size_pyramid": [\n    {\n      "size": "other",\n      "level": "P3",\n      "reason": "raw_none_below_confidence",\n      "count": 21\n    },\n    {\n      "size": "other",\n      "level": "P4",\n      "reason": "raw_none_below_confidence",\n      "count": 103\n    },\n    {\n      "size": "other",\n      "level": "P5",\n      "reason": "raw_none_below_confidence",\n      "count": 1\n    },\n    {\n      "size": "other",\n      "level": "no_iou50",\n      "reason": "no_raw_box_iou50",\n      "count": 49\n    },\n    {\n      "size": "small",\n      "level": "P3",\n      "reason": "raw_none_below_confidence",\n      "count": 6\n    },\n    {\n      "size": "small",\n      "level": "P4",\n      "reason": "raw_none_below_confidence",\n      "count": 13\n    },\n    {\n      "size": "small",\n      "level": "no_iou50",\n      "reason": "no_raw_box_iou50",\n      "count": 211\n    }\n  ],\n  "split": "val",\n  "checkpoint_sha256": "a9dc4b05f3bdf6d1beaf20e9eef855c56438ef4dc6a4c0b08eb2b32628d4224e",\n  "native_metrics": {\n    "precision": 0.0,\n    "recall": 0.0,\n    "mAP50": 0.0,\n    "mAP50_95": 0.0,\n    "F1_native_macro": 0.0,\n    "precision_conf025_macro": 0.0,\n    "recall_conf025_macro": 0.0,\n    "F1_conf025_macro": 0.0\n  },\n  "per_class": [\n    {\n      "class_id": 0,\n      "name": "blue",\n      "precision": 0.0,\n      "recall": 0.0,\n      "F1_native": 0.0,\n      "AP50": 0.0,\n      "AP50_95": 0.0,\n      "precision_conf025": 0.0,\n      "recall_conf025": 0.0,\n      "F1_conf025": 0.0,\n      "support": 258\n    },\n    {\n      "class_id": 1,\n      "name": "white",\n      "precision": 0.0,\n      "recall": 0.0,\n      "F1_native": 0.0,\n      "AP50": 0.0,\n      "AP50_95": 0.0,\n      "precision_conf025": 0.0,\n      "recall_conf025": 0.0,\n      "F1_conf025": 0.0,\n      "support": 349\n    },\n    {\n      "class_id": 2,\n      "name": "yellow",\n      "precision": 0.0,\n      "recall": 0.0,\n      "F1_native": 0.0,\n      "AP50": 0.0,\n      "AP50_95": 0.0,\n      "precision_conf025": 0.0,\n      "recall_conf025": 0.0,\n      "F1_conf025": 0.0,\n      "support": 448\n    },\n    {\n      "class_id": 3,\n      "name": "red",\n      "precision": 0.0,\n      "recall": 0.0,\n      "F1_native": 0.0,\n      "AP50": 0.0,\n      "AP50_95": 0.0,\n      "precision_conf025": 0.0,\n      "recall_conf025": 0.0,\n      "F1_conf025": 0.0,\n      "support": 279\n    },\n    {\n      "class_id": 4,\n      "name": "none",\n      "precision": 0.0,\n      "recall": 0.0,\n      "F1_native": 0.0,\n      "AP50": 0.0,\n      "AP50_95": 0.0,\n      "precision_conf025": 0.0,\n      "recall_conf025": 0.0,\n      "F1_conf025": 0.0,\n      "support": 404\n    }\n  ],\n  "definitions": {\n    "precision_recall": "Ultralytics class means at the native operating point",\n    "F1_native_macro": "Mean class F1 at the native mean-F1 operating point; IoU 0.5",\n    "F1_conf025_macro": "Mean class F1 interpolated from PR/confidence curves at confidence 0.25; IoU 0.5",\n    "AP": "Ultralytics 101-point AP integration; AP50 and mean AP at IoU 0.50:0.05:0.95"\n  }\n}\n', 'treatment/summary.json': b'{\n  "fixed_025": {\n    "tp": 0,\n    "fp": 0,\n    "fn": 404,\n    "precision": null,\n    "recall": 0.0,\n    "threshold": 0.25\n  },\n  "operating_point": null,\n  "images": 316,\n  "support_none": 404,\n  "errors_by_size_pyramid": [\n    {\n      "size": "other",\n      "level": "P3",\n      "reason": "raw_none_below_confidence",\n      "count": 20\n    },\n    {\n      "size": "other",\n      "level": "P4",\n      "reason": "raw_none_below_confidence",\n      "count": 103\n    },\n    {\n      "size": "other",\n      "level": "P5",\n      "reason": "raw_none_below_confidence",\n      "count": 1\n    },\n    {\n      "size": "other",\n      "level": "no_iou50",\n      "reason": "no_raw_box_iou50",\n      "count": 50\n    },\n    {\n      "size": "small",\n      "level": "P3",\n      "reason": "raw_none_below_confidence",\n      "count": 6\n    },\n    {\n      "size": "small",\n      "level": "P4",\n      "reason": "raw_none_below_confidence",\n      "count": 13\n    },\n    {\n      "size": "small",\n      "level": "no_iou50",\n      "reason": "no_raw_box_iou50",\n      "count": 211\n    }\n  ],\n  "split": "val",\n  "checkpoint_sha256": "f8cd51a942524110c111765e58a7b639f5c06c899dd8e32cb2856be4b9c47eb0",\n  "native_metrics": {\n    "precision": 0.0,\n    "recall": 0.0,\n    "mAP50": 0.0,\n    "mAP50_95": 0.0,\n    "F1_native_macro": 0.0,\n    "precision_conf025_macro": 0.0,\n    "recall_conf025_macro": 0.0,\n    "F1_conf025_macro": 0.0\n  },\n  "per_class": [\n    {\n      "class_id": 0,\n      "name": "blue",\n      "precision": 0.0,\n      "recall": 0.0,\n      "F1_native": 0.0,\n      "AP50": 0.0,\n      "AP50_95": 0.0,\n      "precision_conf025": 0.0,\n      "recall_conf025": 0.0,\n      "F1_conf025": 0.0,\n      "support": 258\n    },\n    {\n      "class_id": 1,\n      "name": "white",\n      "precision": 0.0,\n      "recall": 0.0,\n      "F1_native": 0.0,\n      "AP50": 0.0,\n      "AP50_95": 0.0,\n      "precision_conf025": 0.0,\n      "recall_conf025": 0.0,\n      "F1_conf025": 0.0,\n      "support": 349\n    },\n    {\n      "class_id": 2,\n      "name": "yellow",\n      "precision": 0.0,\n      "recall": 0.0,\n      "F1_native": 0.0,\n      "AP50": 0.0,\n      "AP50_95": 0.0,\n      "precision_conf025": 0.0,\n      "recall_conf025": 0.0,\n      "F1_conf025": 0.0,\n      "support": 448\n    },\n    {\n      "class_id": 3,\n      "name": "red",\n      "precision": 0.0,\n      "recall": 0.0,\n      "F1_native": 0.0,\n      "AP50": 0.0,\n      "AP50_95": 0.0,\n      "precision_conf025": 0.0,\n      "recall_conf025": 0.0,\n      "F1_conf025": 0.0,\n      "support": 279\n    },\n    {\n      "class_id": 4,\n      "name": "none",\n      "precision": 0.0,\n      "recall": 0.0,\n      "F1_native": 0.0,\n      "AP50": 0.0,\n      "AP50_95": 0.0,\n      "precision_conf025": 0.0,\n      "recall_conf025": 0.0,\n      "F1_conf025": 0.0,\n      "support": 404\n    }\n  ],\n  "definitions": {\n    "precision_recall": "Ultralytics class means at the native operating point",\n    "F1_native_macro": "Mean class F1 at the native mean-F1 operating point; IoU 0.5",\n    "F1_conf025_macro": "Mean class F1 interpolated from PR/confidence curves at confidence 0.25; IoU 0.5",\n    "AP": "Ultralytics 101-point AP integration; AP50 and mean AP at IoU 0.50:0.05:0.95"\n  }\n}\n'}.items():
    path = smoke_dir / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(content)


In [ ]:
from warm_launch import run_logged

run_logged([sys.executable, str(CODE_DIR / 'workflow.py'), *['pair', '--seed', '3407', '--smoke-receipt', '/kaggle/working/verified_smoke/SMOKE_READY.json']],
           Path('/kaggle/working/yolo11_workflow.log'))
